In [ ]:
N_WORKERS = 6
ACCOUNT_INDEX = 0  # Owner=0, friend 1=1, friend 2=2. Change nothing else.


# 02 — Private Gemma workers (T4 x2)
Attach the private dataset and enable Internet. Add your own HF_TOKEN secret.
Use Save Version -> Save & Run All. This notebook never prints credentials.
Use the provisional copy only on accounts 1 and 2. Its local timing gate
starts its shards, but only account 0's approval can authorize analysis.
No public dataset/notebook/output versions.
Each GPU receives one fresh process and a complete fp16 model. No TPU/P100.

In [ ]:
import sys, json, time, subprocess
from pathlib import Path
SESSION_STARTED = time.monotonic()
runtime = Path("/kaggle/working/counterprior_runtime")
runtime.mkdir(parents=True, exist_ok=True)
payload = {'avh_common.py': '"""CounterPrior v2, independent continuation, November 2026.\n\nImport is CPU-only. Model dependencies are imported only at the inference boundary.\nBulk inference requires GO, terms, a manifest, and formal smoke. Audit gates analysis.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport csv\nimport dataclasses\nimport faulthandler\nimport hashlib\nimport json\nimport os\nimport platform\nimport re\nimport subprocess\nimport threading\nimport time\nimport traceback\nimport uuid\nfrom collections import defaultdict\nfrom pathlib import Path\n\nos.environ.setdefault("MPLBACKEND", "Agg")\nTASKS = ("Video-driven Audio Hallucination", "Audio-driven Video Hallucination", "AV Matching")\nPASSES = ("full", "cf0", "cf1", "cf2", "cf3", "blank", "no_media")\nMODEL_IDS = {"gemma": "google/gemma-4-E2B-it", "qwen": "Qwen/Qwen2.5-Omni-3B"}\nSEED = 20261103\nLINEAGE_LIMITATION = "lineage_unrecoverable: grouping by released clip; swapped clips may share sources across splits"\nOBJECT_RULE = "object-v1: casefold/normalize spaces; extract anchored Is there a/an, Is the ... making sound/visible, Can you hear/see; otherwise exact normalized question"\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for block in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    with open(tmp, "w", encoding="utf-8") as f:\n        json.dump(value, f, indent=2, allow_nan=False)\n        f.flush()\n        os.fsync(f.fileno())\n    os.replace(tmp, path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef require_go(owner_release):\n    if owner_release != "GO":\n        raise RuntimeError("HALT: notebook review required; owner must type GO in chat first.")\n\n\n@dataclasses.dataclass(frozen=True)\nclass Config:\n    root: str\n    model: str\n    revision: str  # resolved immutable Hub SHA, never main\n    frames: int = 8\n    max_side: int = 224\n    seed: int = SEED\n    k: int = 4\n    worker_index: int = 0\n    workers: int = 1\n    stage: str = "full"\n    scoring_mode: str = "verified_prefix"\n\n    def __post_init__(self):\n        if self.model not in MODEL_IDS or self.frames not in (1, 8) or self.k not in (2, 4):\n            raise ValueError("Only the two approved models, 8/1 frames, and preregistered K=2/4 are permitted")\n        if self.stage not in PASSES or not 0 <= self.worker_index < self.workers:\n            raise ValueError("Invalid stage or work partition")\n        if self.stage.startswith("cf") and int(self.stage[2:]) >= self.k:\n            raise ValueError("Counterfactual stage exceeds registered K")\n        if self.scoring_mode not in ("verified_prefix", "full_forward"):\n            raise ValueError("Unknown scoring mode")\n        if len(self.revision) != 40 or any(c not in "0123456789abcdef" for c in self.revision):\n            raise ValueError("Freeze the model to a 40-character Hub commit SHA")\n\n\ndef resolve_model(model):\n    """Save actual license evidence; some repositories link it instead of shipping LICENSE."""\n    from html.parser import HTMLParser\n    from urllib.request import urlopen\n    from huggingface_hub import HfApi, hf_hub_download\n    repo = MODEL_IDS[model]\n    info = HfApi().model_info(repo)\n    files = {s.rfilename for s in info.siblings}\n    candidates = sorted(f for f in files if f.casefold() in ("license", "license.txt", "license.md"))\n    extra = {}\n    if candidates:\n        name = candidates[0]\n        text = Path(hf_hub_download(repo, name, revision=info.sha)).read_text(encoding="utf-8")\n        source_url = f"https://huggingface.co/{repo}/resolve/{info.sha}/{name}"\n    else:\n        card_path = hf_hub_download(repo, "README.md", revision=info.sha)\n        card = Path(card_path).read_text(encoding="utf-8")\n        # This fallback is specific to the publisher\'s explicit Gemma 4 model-card link.\n        source_url = "https://ai.google.dev/gemma/docs/gemma_4_license"\n        if (model != "gemma" or not re.search(r"(?m)^license:\\s*apache-2\\.0\\s*$", card)\n                or not re.search(r"(?m)^license_link:\\s*" + re.escape(source_url) + r"\\s*$", card)):\n            raise RuntimeError("No bundled license or verified publisher license link; inspect model card")\n        class LicenseParser(HTMLParser):\n            def __init__(self):\n                super().__init__()\n                self.depth, self.parts = 0, []\n            def handle_starttag(self, tag, attrs):\n                if tag == "pre":\n                    self.depth += 1\n            def handle_endtag(self, tag):\n                if tag == "pre":\n                    self.depth -= 1\n            def handle_data(self, value):\n                if self.depth > 0:\n                    self.parts.append(value)\n        with urlopen(source_url, timeout=30) as response:\n            resolved = response.geturl()\n            if resolved not in (source_url, "https://ai.google.dev/gemma/apache_2"):\n                raise RuntimeError("Unexpected redirect for publisher license evidence")\n            parser = LicenseParser()\n            parser.feed(response.read().decode("utf-8"))\n        text = "".join(parser.parts).strip()\n        if not all(term in text for term in ("Apache License", "Version 2.0", "Grant of Copyright License")):\n            raise RuntimeError("Publisher page did not contain the expected full license text")\n        extra = {"model_card_text": card, "model_card_sha256": file_hash(card_path),\n                 "model_card_source_url": f"https://huggingface.co/{repo}/resolve/{info.sha}/README.md",\n                 "license_resolved_url": resolved}\n    if not text.strip():\n        raise RuntimeError("License evidence was empty")\n    return {"repo": repo, "revision": info.sha, "license_text": text,\n            "license_sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),\n            "license_source_url": source_url, **extra}\n\n\ndef download_release(root, owner_release):\n    """Post-GO only. Archives stay private. No automatic extraction of untrusted paths."""\n    require_go(owner_release)\n    import gdown\n    root = Path(root) / "release"\n    root.mkdir(parents=True, exist_ok=True)\n    ids = {"videos": "10-Qp8zxA3ITT-ileEnCgJkf5Nzx1wry7",\n           "qa": "1KcYDAv9lLy3hsx5rWdfRqMFV2NYcZ94W"}\n    for name, drive_id in ids.items():\n        target = root / (name + ".download")\n        if not target.exists():\n            tmp = target.with_suffix(".partial")\n            result = gdown.download(id=drive_id, output=str(tmp), quiet=False)\n            if not result:\n                raise RuntimeError("Dataset download failed; STOP rather than substitute data")\n            os.replace(tmp, target)\n    print("Downloaded private release files; inspect archive members and README/terms before extraction/use.")\n    return {p.name: file_hash(p) for p in root.glob("*.download")}\n\n\ndef extract_release(root):\n    """Extract only regular files/directories in zip/tar; reject traversal and symlinks."""\n    import tarfile\n    import zipfile\n    root = Path(root) / "release"\n    out = (root / "unpacked").resolve()\n    out.mkdir(exist_ok=True)\n    for archive in root.glob("*.download"):\n        def safe(name):\n            dest = (out / name).resolve()\n            if not dest.is_relative_to(out):\n                raise ValueError("Unsafe archive member: " + name)\n            return dest\n        if zipfile.is_zipfile(archive):\n            with zipfile.ZipFile(archive) as z:\n                for member in z.infolist():\n                    safe(member.filename)\n                    if (member.external_attr >> 16) & 0o170000 == 0o120000:\n                        raise ValueError("Archive symlink rejected")\n                z.extractall(out)\n        elif tarfile.is_tarfile(archive):\n            with tarfile.open(archive) as t:\n                for member in t.getmembers():\n                    safe(member.name)\n                    if not (member.isfile() or member.isdir()):\n                        raise ValueError("Archive links/devices rejected")\n                t.extractall(out, filter="data")\n        else:\n            # QA may be a JSON file rather than an archive.\n            value = read_json(archive)\n            atomic_json(out / (archive.stem + ".json"), value)\n    return out\n\n\ndef inventory_release(unpacked, schema_mapping=None, records_key=None, task_mapping=None):\n    """Print unfiltered records first. Noncanonical schemas require explicit mappings."""\n    root = Path(unpacked)\n    media = sorted(root.rglob("*.mp4"))\n    raw, containers = [], {}\n    for path in sorted(root.rglob("*.json")):\n        value = read_json(path)\n        if isinstance(value, dict):\n            containers[str(path.relative_to(root))] = sorted(value)\n            if records_key is not None:\n                value = value.get(records_key)\n            else:\n                # Show the container verbatim; do not silently choose a list-valued key.\n                raw.append(value)\n                continue\n        if isinstance(value, list):\n            raw.extend(value)\n    print("FIRST THREE RAW RECORDS (unmodified field values):")\n    for record in raw[:3]:\n        print(json.dumps(record, ensure_ascii=False, indent=2))\n    mapping = schema_mapping or {k: k for k in ("video_id", "task", "text", "label")}\n    task_key = mapping.get("task", "task")\n    names = sorted({str(x[task_key]) for x in raw if isinstance(x, dict) and task_key in x})\n    terms = [str(p.relative_to(root)) for p in root.rglob("*")\n             if p.is_file() and any(s in p.name.lower() for s in ("readme", "licen", "terms"))]\n    result = {"media_files": len(media), "raw_record_count": len(raw), "terms_files": terms,\n              "all_key_names": sorted({k for x in raw if isinstance(x, dict) for k in x}),\n              "container_key_names": containers, "all_task_names": names,\n              "raw_task_counts": {t: sum(isinstance(x, dict) and str(x.get(task_key)) == t for x in raw) for t in names},\n              "schema_mapping": mapping, "records_key": records_key, "task_mapping": task_mapping or {}}\n    print(json.dumps(result, indent=2))\n    if (set(mapping) != {"video_id", "task", "text", "label"} or not raw or\n            any(not isinstance(x, dict) or any(k not in x for k in mapping.values()) for x in raw)):\n        raise ValueError("QA schema differs: set explicit schema_mapping / records_key from the printed records; no guessing")\n    qa = []\n    for record in raw:\n        row = dict(record)  # preserve release metadata for lineage recovery\n        row.update({canonical: record[actual] for canonical, actual in mapping.items()})\n        row["video_id"] = str(row["video_id"])\n        row["task"] = (task_mapping or {}).get(row["task"], row["task"])\n        qa.append(row)\n    result.update(qa_rows=len(qa), task_counts={t: sum(x["task"] == t for x in qa) for t in sorted({x["task"] for x in qa})})\n    atomic_json(root.parent / "qa_schema_mapping.json", {k: result[k] for k in ("schema_mapping", "records_key", "task_mapping")})\n    print("EXPLICIT QA MAPPING:", json.dumps({k: result[k] for k in ("schema_mapping", "records_key", "task_mapping")}))\n    return qa, result\n\n\ndef check_terms(root, model):\n    """Evidence must be a saved local document + exact applicable passage, not a boolean."""\n    review = read_json(Path(root) / "terms_review.json")\n    for key in ("dataset", model):\n        entry = review[key]\n        private = key == "dataset" and entry.get("decision") == "no_licence_stated_private_research_eval"\n        if entry.get("decision") != "research_use_permitted" and not private:\n            raise RuntimeError(f"STOP: {key} terms not established as usable")\n        evidence = Path(root) / entry["evidence_file"]\n        if file_hash(evidence) != entry["sha256"]:\n            raise RuntimeError("Terms evidence changed")\n        quote = entry.get("applicable_passage", "")\n        if not private and (not quote.strip() or quote not in evidence.read_text(encoding="utf-8")):\n            raise RuntimeError("Record an exact applicable terms passage and source URL")\n        if not entry.get("source_url") or not entry.get("reviewer"):\n            raise RuntimeError("Terms require provenance and a named reviewer")\n    return review\n\n\ndef save_approved_private_terms(root, private_eval_approved=False, reviewer=""):\n    """Record the owner\'s explicit private-only choice with actual saved evidence."""\n    if private_eval_approved is not True or not reviewer.strip():\n        raise RuntimeError("Explicit owner private-evaluation approval and reviewer required")\n    root = Path(root)\n    inventory_path = root / "release_inventory.json"\n    inventory = read_json(inventory_path)\n    if inventory["terms_files"]:\n        raise RuntimeError("Release terms files were found; inspect them before using the no-license decision")\n    license_files = {}\n    review = {"dataset": {"decision": "no_licence_stated_private_research_eval",\n        "evidence_file": inventory_path.name, "sha256": file_hash(inventory_path),\n        "applicable_passage": "", "source_url": "https://github.com/kaist-ami/AVHBench",\n        "reviewer": reviewer, "terms_files_found": inventory["terms_files"],\n        "note": "Owner-approved private research evaluation; not a claim of publisher permission"}}\n    for model in MODEL_IDS:\n        meta = read_json(root / model / "model_metadata.json")\n        if meta["repo"] != MODEL_IDS[model]:\n            raise RuntimeError("Saved model metadata has the wrong repository")\n        text = meta["license_text"]\n        if model == "gemma":\n            valid = "Apache License" in text and "Version 2.0" in text\n            pattern = r"2\\. Grant of Copyright License\\..*?(?=\\n\\s*3\\.)"\n        else:\n            valid = ("Qwen RESEARCH LICENSE" in text and "FOR NON-COMMERCIAL PURPOSES ONLY" in text\n                     and "research or evaluation purposes only" in text)\n            pattern = r"2\\. Grant of Rights.*?(?=\\n\\s*3\\.)"\n        match = re.search(pattern, text, re.S)\n        if not valid or match is None:\n            raise RuntimeError(f"Inspect actual saved {model} license: expected research-use evidence missing")\n        evidence = root / f"{model}_LICENSE.txt"\n        expected_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()\n        if expected_hash != meta["license_sha256"]:\n            raise RuntimeError("Saved model license hash differs from resolved metadata")\n        license_files[evidence] = text\n        source = meta.get("license_source_url") or (\n            f"https://huggingface.co/{meta[\'repo\']}/resolve/{meta[\'revision\']}/LICENSE")\n        review[model] = {"decision": "research_use_permitted", "evidence_file": evidence.name,\n            "sha256": expected_hash, "applicable_passage": match[0], "source_url": source,\n            "reviewer": "Saved-license evidence check; owner authorized private research evaluation",\n            "model_revision": meta["revision"]}\n    target = root / "terms_review.json"\n    if not target.exists() or read_json(target) != review:\n        if any(path for model in MODEL_IDS for path in (root / model).glob("f*/*/*.jsonl")):\n            raise RuntimeError("Terms changed after inference began; preserve existing run identity")\n    for evidence, text in license_files.items():\n        evidence.write_bytes(text.encode("utf-8"))\n    atomic_json(target, review)\n    atomic_json(root / "publication_policy.json", {"dataset_evaluation": "private_only",\n        "public_avhbench_media_allowed": False, "public_content": ["aggregate numbers", "our own text"],\n        "owner_approval": reviewer})\n    for model in MODEL_IDS:\n        check_terms(root, model)\n    return review\n\n\ndef media_catalog(media_dir):\n    """Index filenames once; repeated Drive-wide rglob per clip is prohibitively slow."""\n    result = {"video": defaultdict(list), "metadata": defaultdict(list)}\n    for path in Path(media_dir).rglob("*"):\n        kind = "video" if path.suffix.lower() == ".mp4" else "metadata" if path.suffix.lower() == ".json" else None\n        if kind is not None:\n            result[kind][path.stem].append(path)\n    return result\n\n\ndef object_key(question):\n    text = " ".join(question.casefold().strip().rstrip("?.!").split())\n    patterns = (r"^is there (?:an? |the )?(.+?)(?: (?:in|on) (?:the )?(?:audio|video|image|clip))?$",\n                r"^is (?:the |an? )?(.+?) (?:making (?:a )?sound|visible|audible)(?: .*)?$",\n                r"^(?:can|do) you (?:hear|see) (?:the |an? )?(.+?)(?: (?:in|on) .*)?$")\n    for pattern in patterns:\n        match = re.fullmatch(pattern, text)\n        if match:\n            return match[1].strip()\n    return "question:" + text\n\n\ndef auto_lineage(root, qa, media_dir, catalog=None):\n    """Recover only explicit metadata/marked filename origins; otherwise disclose clip fallback."""\n    path = Path(root) / "lineage.json"\n    if path.exists():\n        return read_json(path)\n    catalog = media_catalog(media_dir) if catalog is None else catalog\n    by_video = defaultdict(list)\n    for row in qa:\n        by_video[str(row["video_id"])].append(row)\n    output = []\n    for vid, rows in sorted(by_video.items()):\n        matches = catalog["video"].get(vid, [])\n        name = str(matches[0].relative_to(media_dir)) if len(matches) == 1 else vid\n        metadata_rows = list(rows)\n        metadata_rows.extend(r["metadata"] for r in rows if isinstance(r.get("metadata"), dict))\n        for sidecar in catalog["metadata"].get(vid, []):\n            value = read_json(sidecar)\n            if isinstance(value, dict):\n                metadata_rows.append(value)\n        origins, evidence = set(), []\n        swapped = any(r.get("is_swapped") is True for r in metadata_rows) or bool(re.search(r"(?:^|[/_\\-])(swapped|swap|synthetic)(?:[/_\\-]|$)", name, re.I))\n        marked = swapped or any(isinstance(r.get("is_swapped"), bool) for r in metadata_rows)\n        for row in metadata_rows:\n            source_ids = row.get("source_ids", [])\n            if isinstance(source_ids, list):\n                origins.update(str(s) for s in source_ids if s is not None and str(s).strip())\n                if source_ids:\n                    evidence.append("metadata:source_ids")\n            for field in ("audio_source_id", "video_source_id", "source_video_id", "valor_id", "audiocaps_id"):\n                if row.get(field) is not None and str(row[field]).strip():\n                    origins.add(str(row[field]))\n                    evidence.append("metadata:" + field)\n        # Explicitly tagged names only, e.g. audio_dog1__video_clip2; never split opaque IDs.\n        tagged = re.fullmatch(r"audio_(.+?)__video_(.+)", vid, re.I)\n        if tagged:\n            origins.update(tagged.groups())\n            evidence.append("filename:audio_<id>__video_<id>")\n            swapped, marked = True, True\n        unrecoverable = not origins or (swapped and len(origins) < 2)\n        output.append({"video_id": vid, "source_ids": sorted(origins) if origins else [vid],\n            "is_swapped": swapped, "swap_status_known": marked,\n            "lineage_unrecoverable": unrecoverable, "grouping_fallback": unrecoverable,\n            "provenance": "; ".join(sorted(set(evidence))) if evidence else "released clip ID fallback",\n            "object_rule": OBJECT_RULE,\n            "object_by_question": {r["text"]: object_key(r["text"]) for r in rows if r["task"] in TASKS}})\n    atomic_json(path, output)\n    atomic_json(Path(root) / "lineage_rules.json", {"object_rule": OBJECT_RULE,\n        "source_fields": ["source_ids", "audio_source_id", "video_source_id", "source_video_id", "valor_id", "audiocaps_id"],\n        "filename_pattern": "audio_<id>__video_<id>",\n        "unknown_swap_status": "not marked swapped, not proof of real origin", "limitation": LINEAGE_LIMITATION})\n    return output\n\n\ndef connected_groups(lineage):\n    """Union all audio/video origins, including swapped clips, before assigning splits."""\n    parent = {}\n    def find(x):\n        parent.setdefault(x, x)\n        if parent[x] != x:\n            parent[x] = find(parent[x])\n        return parent[x]\n    def union(a, b):\n        a, b = find(a), find(b)\n        parent[max(a, b)] = min(a, b)\n    for row in lineage:\n        origins = [row["video_id"]] if row.get("grouping_fallback") else row["source_ids"]\n        if not origins or any(not str(x).strip() for x in origins):\n            raise ValueError("Unknown source lineage: STOP; numeric release IDs are not source IDs")\n        for source in origins:\n            union(str(origins[0]), str(source))\n    return {r["video_id"]: find(str(r["video_id"] if r.get("grouping_fallback") else r["source_ids"][0])) for r in lineage}\n\n\ndef resolve_released_qa(root, qa):\n    """Keep first identical-label copy; exclude all conflicting-label occurrences.\n\n    Apply before manifest freeze. Original release is never edited, and every\n    affected source-row index plus label is preserved in an explicit resolution log.\n    """\n    groups = defaultdict(list)\n    for i, row in enumerate(qa):\n        if row["task"] in TASKS:\n            groups[(str(row["video_id"]), row["task"], row["text"])].append(i)\n    removed, decisions = set(), []\n    for key, indices in groups.items():\n        if len(indices) == 1:\n            continue\n        labels = {str(qa[i]["label"]).strip().casefold() for i in indices}\n        excluded = indices if len(labels) > 1 else indices[1:]\n        removed.update(excluded)\n        decisions.append({"video_id": key[0], "task": key[1], "question": key[2],\n            "source_indices": indices, "source_labels": [qa[i]["label"] for i in indices],\n            "removed_indices": excluded,\n            "action": "exclude_conflicting_group" if len(labels) > 1 else "keep_first_identical_label_copy"})\n    resolved = [r for i, r in enumerate(qa) if i not in removed]\n    report = {"rule": "qa-dedup-v1: keep first identical-label copy; exclude every conflicting-label occurrence",\n        "raw_qa_sha256": digest(qa), "resolved_qa_sha256": digest(resolved),\n        "raw_records": len(qa), "resolved_records": len(resolved),\n        "raw_binary_questions": sum(r["task"] in TASKS for r in qa),\n        "resolved_binary_questions": sum(r["task"] in TASKS for r in resolved),\n        "removed_occurrences": len(removed), "decisions": decisions}\n    target = Path(root) / "qa_resolution.json"\n    if target.exists() and read_json(target) != report:\n        raise RuntimeError("QA resolution changed; do not replace frozen study records")\n    atomic_json(target, report)\n    return resolved, report\n\n\ndef make_manifest(root, qa, lineage, seed=SEED, qa_resolution=None):\n    """lineage: video_id, source_ids (both origins), is_swapped, provenance, object map.\n\n    Auto object keys and explicitly disclosed released-clip fallback are supported.\n    """\n    import numpy as np\n    root = Path(root)\n    groups = connected_groups(lineage)\n    by_video = {r["video_id"]: r for r in lineage}\n    if len(by_video) != len(lineage):\n        raise ValueError("Duplicate lineage IDs")\n    unique = sorted(set(groups.values()))\n    if len(unique) < 10:\n        raise ValueError("Fewer than ten independent connected source groups: STOP for protocol review")\n    order = np.random.default_rng(seed).permutation(unique).tolist()\n    ncal, nval = max(1, int(.6 * len(order))), max(1, int(.2 * len(order)))\n    splits = {g: ("calibration" if i < ncal else "validation" if i < ncal+nval else "test")\n              for i, g in enumerate(order)}\n    rows = []\n    for q in qa:\n        if q["task"] not in TASKS:\n            continue\n        vid = str(q["video_id"])\n        meta = by_video[vid]\n        if not meta.get("provenance") or not isinstance(meta.get("is_swapped"), bool):\n            raise ValueError("Record lineage provenance and real/swapped status")\n        label = str(q["label"]).strip().lower()\n        if label not in ("yes", "no"):\n            raise ValueError("Unexpected yes/no label")\n        obj = meta["object_by_question"][q["text"]]\n        if not obj:\n            raise ValueError("Object/event key must be derived before scoring")\n        row = {"id": digest([vid, q["task"], q["text"]])[:24], "video_id": vid,\n               "source_group": groups[vid], "source_ids": meta["source_ids"],\n               "split": splits[groups[vid]], "task": q["task"], "question": q["text"],\n               "label": int(label == "yes"), "object": obj, "is_swapped": meta["is_swapped"],\n               "swap_status_known": meta.get("swap_status_known", True)}\n        rows.append(row)\n    if len({r["id"] for r in rows}) != len(rows):\n        raise ValueError("Duplicate QA rows; resolve explicitly, do not silently deduplicate")\n    # Donor selection is clip-level, unique, split-local, and source-disjoint from target.\n    for row in rows:\n        candidates = sorted({r["video_id"] for r in rows if r["split"] == row["split"]\n                             and r["source_group"] != row["source_group"] and not r["is_swapped"]})\n        if len(candidates) < 4:\n            raise ValueError("Need four real, other-source donor clips in every split")\n        rng = np.random.default_rng(int(digest([seed, row["video_id"], row["task"]])[:16], 16))\n        row["donors"] = rng.choice(candidates, 4, replace=False).tolist()\n    path = root / "manifest.json"\n    limitations = [LINEAGE_LIMITATION] if any(r.get("lineage_unrecoverable") for r in lineage) else []\n    if any(r.get("swap_status_known") is False for r in lineage):\n        limitations.append("Swap status absent for some clips: donor pool uses clips not marked swapped; real origin is unverified for those clips")\n    payload = {"seed": seed, "rows": rows, "lineage_sha256": digest(lineage),\n               "qa_sha256": digest(qa), "split_rule": "connected recovered origins or disclosed released-clip fallback, 60/20/20",\n               "limitations": limitations, "object_rule": OBJECT_RULE}\n    if qa_resolution is not None:\n        payload["qa_resolution"] = qa_resolution\n    if path.exists() and read_json(path) != payload:\n        raise RuntimeError("Manifest already frozen: start a new study directory to change it")\n    atomic_json(path, payload)\n    return payload\n\n\ndef audit_targets(manifest, limit=100):\n    selected = {}\n    label_items = sorted((r for r in manifest["rows"] if r["is_swapped"] or r.get("swap_status_known") is False),\n                         key=lambda r: digest([SEED, "audit", r["id"]]))[:limit]\n    objects = sorted(manifest["rows"], key=lambda r: digest([SEED, "object-check", r["id"]]))[:50]\n    for row in label_items + objects:\n        selected[row["id"]] = dict(row, label_audit=row in label_items, object_spotcheck=row in objects)\n    return list(selected.values())\n\n\ndef create_audit(root, manifest, limit=100):\n    """Blind media/label review; no model answers or hidden states in this worksheet."""\n    root = Path(root)\n    path = root / "audit.csv"\n    if path.exists():\n        return path\n    items = audit_targets(manifest, limit)\n    fields = ["id", "video_id", "task", "question", "label", "object", "label_audit", "object_spotcheck",\n              "decision", "object_check", "reviewer", "note"]\n    with open(path, "w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows({k: r.get(k, "") for k in fields} for r in items)\n    return path\n\n\ndef check_audit(root, manifest, require_complete=False):\n    path = Path(root) / "audit.csv"\n    if not path.exists() and not require_complete:\n        return set()\n    with open(path, encoding="utf-8", newline="") as f:\n        rows = list(csv.DictReader(f))\n    expected = audit_targets(manifest)\n    if {r["id"] for r in rows} != {r["id"] for r in expected} or len(rows) != len(expected):\n        raise RuntimeError("Audit selection changed or incomplete")\n    targets = {r["id"]: r for r in expected}\n    complete = [r for r in rows if targets[r["id"]]["label_audit"] and\n                r["decision"] in ("valid", "exclude", "uncertain") and r["reviewer"]]\n    object_complete = [r for r in rows if targets[r["id"]]["object_spotcheck"] and\n                       r.get("object_check") in ("correct", "flagged") and r["reviewer"]]\n    n_labels = sum(r["label_audit"] for r in expected)\n    n_objects = sum(r["object_spotcheck"] for r in expected)\n    if require_complete and (len(complete) != n_labels or len(object_complete) != n_objects):\n        raise RuntimeError("Complete label audit and 50-item object spot-check before analysis freeze")\n    excluded = {r["id"] for r in complete if r["decision"] != "valid"}\n    # Audit error CI clusters by source, using the shared bootstrap implementation.\n    groups = {r["id"]: r["source_group"] for r in manifest["rows"]}\n    import numpy as np\n    error = np.array([r["decision"] == "exclude" for r in complete], dtype=float)\n    result = cluster_ci(error, [groups[r["id"]] for r in complete], lambda a: float(a.mean()))\n    result.update({"n_audited": len(complete), "n_label_pending": n_labels-len(complete),\n                   "n_object_checked": len(object_complete), "n_object_pending": n_objects-len(object_complete),\n                   "n_object_flagged": sum(r["object_check"] == "flagged" for r in object_complete),\n                   "n_uncertain": sum(r["decision"] == "uncertain" for r in complete),\n                   "excluded_ids": sorted(excluded), "audit_sha256": file_hash(path)})\n    atomic_json(Path(root) / "audit_summary.json", result)\n    return excluded\n\n\ndef prepare_media(root, video_dir, manifest, frames=8, max_side=224, catalog=None):\n    """Same deterministic frame indices/audio bytes for both models; full track, no truncation."""\n    import numpy as np\n    from PIL import Image\n    import av\n    import soundfile as sf\n    root = Path(root)\n    video_dir = Path(video_dir)\n    catalog = media_catalog(video_dir) if catalog is None else catalog\n    index = {}\n    videos = sorted({r["video_id"] for r in manifest["rows"]})\n    for number, vid in enumerate(videos, 1):\n        matches = catalog["video"].get(vid, [])\n        if len(matches) != 1:\n            raise ValueError(f"Need exactly one media file for {vid}, found {len(matches)}")\n        source = matches[0]\n        target = root / "media" / f"f{frames}-s{max_side}" / vid\n        target.mkdir(parents=True, exist_ok=True)\n        ready = target / "ready.json"\n        if ready.exists():\n            saved = read_json(ready)\n            if saved["source_sha256"] != file_hash(source):\n                raise RuntimeError("Source video changed during preprocessing")\n            if any(file_hash(target / name) != sha for name, sha in saved["file_hashes"].items()):\n                raise RuntimeError("Preprocessed media changed")\n            index[vid] = saved\n            if number % 50 == 0 or number == len(videos):\n                print(f"Media ready: {number}/{len(videos)} (verified/reused)", flush=True)\n            continue\n        with av.open(str(source)) as container:\n            count = sum(1 for _ in container.decode(video=0))\n        if count < 1:\n            raise ValueError("Empty video")\n        indices = (np.linspace(0, count - 1, frames).round().astype(int).tolist()\n                   if frames == 8 else [count // 2])\n        with av.open(str(source)) as container:\n            for i, frame in enumerate(container.decode(video=0)):\n                for j in [j for j, wanted in enumerate(indices) if wanted == i]:\n                    image = frame.to_image().convert("RGB")\n                    image.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)\n                    image.save(target / f"{j:02d}.png")\n        subprocess.run(["ffmpeg", "-nostdin", "-y", "-v", "error", "-i", str(source),\n                        "-vn", "-ac", "1", "-ar", "16000", str(target / "audio.wav")], check=True)\n        audio, sr = sf.read(target / "audio.wav")\n        if sr != 16000 or audio.size == 0 or not np.isfinite(audio).all():\n            raise ValueError("Invalid extracted audio")\n        index[vid] = {"directory": str(target), "source_sha256": file_hash(source),\n                      "frame_indices": indices, "audio_samples": len(audio),\n                      "file_hashes": {p.name: file_hash(p) for p in sorted(target.iterdir()) if p.name != "ready.json"}}\n        atomic_json(ready, index[vid])\n        if number % 50 == 0 or number == len(videos):\n            print(f"Media ready: {number}/{len(videos)}", flush=True)\n    atomic_json(root / f"media_index_f{frames}.json", index)\n    return index\n\n\ndef prepare_shared_study(root, private_eval_approved=False, reviewer="", frames=8, max_side=224):\n    """Run on Account A only. Prepare shared evidence without touching either model."""\n    root = Path(root)\n    print("Recording approved private-only terms and model license evidence...", flush=True)\n    save_approved_private_terms(root, private_eval_approved, reviewer)\n    print("Indexing released media filenames once...", flush=True)\n    unpacked = root / "release" / "unpacked"\n    catalog = media_catalog(unpacked)\n    qa = read_json(root / "released_qa.json")\n    lineage = auto_lineage(root, qa, unpacked, catalog=catalog)\n    resolved, resolution = resolve_released_qa(root, qa)\n    print(json.dumps({"raw_binary_questions": resolution["raw_binary_questions"],\n        "resolved_binary_questions": resolution["resolved_binary_questions"],\n        "removed_occurrences": resolution["removed_occurrences"],\n        "resolution_log": str(root / "qa_resolution.json")}, indent=2), flush=True)\n    manifest = make_manifest(root, resolved, lineage, qa_resolution=resolution)\n    audit_path = create_audit(root, manifest)\n    counts = {s: sum(r["split"] == s for r in manifest["rows"])\n              for s in ("calibration", "validation", "test")}\n    print(json.dumps({"manifest_questions": len(manifest["rows"]), "split_counts": counts,\n                      "limitations": manifest["limitations"], "audit_sheet": str(audit_path)}, indent=2), flush=True)\n    print("Preparing shared frames + 16 kHz audio; per-clip checkpoints permit resume...", flush=True)\n    index = prepare_media(root, unpacked, manifest, frames, max_side, catalog=catalog)\n    summary = {"status": "shared_data_ready", "manifest_questions": len(manifest["rows"]),\n        "media_clips": len(index), "split_counts": counts, "frames": frames,\n        "audit_sheet": str(audit_path), "audit_completion": "required at analysis freeze, not inference",\n        "dataset_decision": "no_licence_stated_private_research_eval", "public_avhbench_media_allowed": False,\n        "limitations": manifest["limitations"], "manifest_sha256": digest(manifest)}\n    summary["qa_resolution"] = {k: resolution[k] for k in\n        ("raw_binary_questions", "resolved_binary_questions", "removed_occurrences", "rule")}\n    atomic_json(root / "shared_data_summary.json", summary)\n    return manifest, index, summary\n\n\ndef verify_media(index):\n    for entry in index.values():\n        directory = Path(entry["directory"])\n        for name, sha in entry["file_hashes"].items():\n            if file_hash(directory / name) != sha:\n                raise RuntimeError("Media bytes changed since preparation")\n\n\nclass Watchdog:\n    """A truly wedged CUDA call cannot be interrupted safely: dump stacks, end kernel.\n\n    Only active inside an inference stage; no background idle-kernel termination.\n    """\n    def __init__(self, seconds=600, deadline=None):\n        self.seconds = seconds\n        self.deadline = deadline\n        self.last = time.monotonic()\n        self.stop = threading.Event()\n\n    def touch(self):\n        self.last = time.monotonic()\n\n    def __enter__(self):\n        def monitor():\n            while not self.stop.wait(5):\n                now = time.monotonic()\n                if now - self.last > self.seconds or (self.deadline is not None and now > self.deadline):\n                    try:\n                        print("STALLED: no durable row within time limit; terminating kernel", flush=True)\n                        # IPython\'s stderr wrapper may not implement fileno().\n                        # Use the real descriptor; diagnostics must never disable termination.\n                        faulthandler.dump_traceback(file=2, all_threads=True)\n                    finally:\n                        os._exit(124)\n        self.thread = threading.Thread(target=monitor, daemon=True)\n        self.thread.start()\n        return self\n\n    def __exit__(self, *args):\n        self.stop.set()\n        self.thread.join(timeout=6)\n\n\nclass Journal:\n    """Single writer, dedicated partition. Per-question features durable before row commit.\n\n    Unique stage/partition paths avoid cross-account shared-file appends. Drive is not\n    a transactional filesystem: never run the same partition twice concurrently.\n    """\n    def __init__(self, directory, worker, signature):\n        self.directory = Path(directory)\n        self.directory.mkdir(parents=True, exist_ok=True)\n        self.path = self.directory / f"{worker}.jsonl"\n        self.lock = self.directory / f"{worker}.lock"\n        self.signature = signature\n\n    def __enter__(self):\n        fd = os.open(self.lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)\n        with os.fdopen(fd, "w") as f:\n            json.dump({"pid": os.getpid(), "started": time.time(), "signature": self.signature}, f)\n        self.rows = []\n        if self.path.exists():\n            raw = self.path.read_bytes()\n            lines = raw.splitlines(keepends=True)\n            valid_bytes = 0\n            for i, line in enumerate(lines):\n                try:\n                    row = json.loads(line)\n                except (ValueError, UnicodeDecodeError):\n                    if i != len(lines)-1 or line.endswith(b"\\n"):\n                        self.lock.unlink()\n                        raise RuntimeError("Corrupt journal interior; preserve and investigate")\n                    break  # only an incomplete final write can be recovered\n                if row["signature"] != self.signature:\n                    self.lock.unlink()\n                    raise RuntimeError("Journal configuration mismatch")\n                self.rows.append(row)\n                valid_bytes += len(line)\n            with open(self.path, "r+b") as f:\n                f.truncate(valid_bytes)\n            if self.rows and not self.path.read_bytes().endswith(b"\\n"):\n                with open(self.path, "ab") as f:\n                    f.write(b"\\n")\n        self.done = {r["id"] for r in self.rows}\n        if len(self.done) != len(self.rows):\n            self.lock.unlink()\n            raise RuntimeError("Duplicate journal IDs")\n        return self\n\n    def append(self, row, hidden):\n        import numpy as np\n        if row["id"] in self.done:\n            return\n        if hidden is not None:\n            target = self.directory / "pending_features" / (row["id"] + ".npz")\n            target.parent.mkdir(exist_ok=True)\n            tmp = target.with_suffix(".tmp")\n            with open(tmp, "wb") as f:\n                np.savez(f, hidden=np.asarray(hidden, dtype=np.float16))\n                f.flush()\n                os.fsync(f.fileno())\n            os.replace(tmp, target)\n            row["feature"] = str(target.relative_to(self.directory))\n            row["feature_sha256"] = file_hash(target)\n        row["signature"] = self.signature\n        with open(self.path, "a", encoding="utf-8") as f:\n            f.write(json.dumps(row, allow_nan=False) + "\\n")\n            f.flush()\n            os.fsync(f.fileno())\n        self.rows.append(row)\n        self.done.add(row["id"])\n\n    def compact(self):\n        import numpy as np\n        rows = [r for r in self.rows if "feature" in r]\n        for start in range(0, len(rows), 200):\n            batch = rows[start:start+200]\n            path = self.directory / f"{self.path.stem}-features-{start//200:05d}.npz"\n            # Pending per-item files remain the recovery source. Shards are derived.\n            states = [np.load(self.directory / r["feature"])["hidden"] for r in batch]\n            with open(path.with_suffix(".tmp"), "wb") as f:\n                np.savez(f, ids=np.asarray([r["id"] for r in batch]), hidden=np.stack(states).astype(np.float16))\n            os.replace(path.with_suffix(".tmp"), path)\n\n    def __exit__(self, *args):\n        self.lock.unlink(missing_ok=True)\n\n\ndef attention_backend(model):\n    # Gemma\'s chunked audio attention only implements eager in Transformers.\n    return ({"": "sdpa", "text_config": "sdpa", "vision_config": "sdpa", "audio_config": "eager"}\n            if model == "gemma" else "sdpa")\n\n\nclass OmniScorer:\n    def __init__(self, cfg):\n        import torch\n        import transformers\n        self.torch, self.cfg = torch, cfg\n        if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):\n            raise RuntimeError("Protocol requires a CUDA T4; do not claim fit from CPU/offload")\n        if torch.cuda.get_device_properties(0).total_memory < 14 * 1024**3:\n            raise RuntimeError("Insufficient GPU memory")\n        torch.manual_seed(cfg.seed)\n        torch.cuda.reset_peak_memory_stats()\n        self.processor = transformers.AutoProcessor.from_pretrained(MODEL_IDS[cfg.model], revision=cfg.revision)\n        cls = (transformers.Gemma4ForConditionalGeneration if cfg.model == "gemma"\n               else transformers.Qwen2_5OmniThinkerForConditionalGeneration)\n        self.model = cls.from_pretrained(MODEL_IDS[cfg.model], revision=cfg.revision,\n                    torch_dtype=torch.float16, device_map={"": 0},\n                    attn_implementation=attention_backend(cfg.model)).eval()\n        if any(p.device.type != "cuda" or (p.is_floating_point() and p.dtype != torch.float16)\n               for p in self.model.parameters()):\n            raise RuntimeError("All parameters must be fp16 on GPU; no silent offload/quantization")\n        # v4 Qwen projects every sequence position otherwise. Last-token logits only;\n        # decoder hidden-state outputs remain unchanged. Also used in generation parity.\n        self.head_hook = self.model.get_output_embeddings().register_forward_pre_hook(\n            lambda module, args: (args[0][:, -1:, :],))\n        self.tokens = self.answer_tokens()\n        self.prefix = None\n        self.prefix_key = None\n        self.env = {"python": platform.python_version(), "torch": torch.__version__,\n                    "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name(0),\n                    "model_revision": cfg.revision, "cuda": torch.version.cuda,\n                    "attention_backend": attention_backend(cfg.model),\n                    "scoring_mode": cfg.scoring_mode,\n                    "packages": subprocess.check_output([os.sys.executable, "-m", "pip", "freeze"], text=True).splitlines()}\n        print(json.dumps({"environment": self.env, "answer_tokens": self.tokens}, indent=2))\n\n    def answer_tokens(self):\n        tok = self.processor.tokenizer\n        result = {"yes": [], "no": []}\n        # Enumerate every vocabulary token decoding to exactly a Yes/No case/space variant.\n        for token in range(len(tok)):\n            if token in tok.all_special_ids:\n                continue\n            decoded = tok.decode([token], skip_special_tokens=False)\n            if decoded.strip().casefold() in result:\n                result[decoded.strip().casefold()].append(token)\n        if not all(result.values()) or set(result["yes"]) & set(result["no"]):\n            raise RuntimeError("Invalid Yes/No token mapping")\n        return result\n\n    def media(self, row, pass_name, index):\n        import numpy as np\n        import soundfile as sf\n        from PIL import Image\n        if pass_name == "no_media":\n            return None, None, "no_media"\n        vid = row["video_id"]\n        image_vid = audio_vid = vid\n        if pass_name.startswith("cf"):\n            donor = row["donors"][int(pass_name[-1])]\n            if row["task"] == TASKS[0]:\n                audio_vid = donor\n            elif row["task"] == TASKS[1]:\n                image_vid = donor\n            else:\n                raise ValueError("AV Matching has full/no-media only; no privileged queried sense")\n        images = [Image.open(p).convert("RGB") for p in sorted(Path(index[image_vid]["directory"]).glob("*.png"))]\n        audio, sr = sf.read(Path(index[audio_vid]["directory"]) / "audio.wav", dtype="float32")\n        if sr != 16000 or len(images) != self.cfg.frames:\n            raise ValueError("Media format mismatch")\n        if pass_name == "blank":\n            if row["task"] == TASKS[0]:\n                audio = np.zeros_like(audio)\n            elif row["task"] == TASKS[1]:\n                images = [Image.new("RGB", im.size) for im in images]\n            else:\n                raise ValueError("AV Matching blank undefined")\n        return images, audio, digest([image_vid, audio_vid,\n                    row["task"] if pass_name == "blank" else None, self.cfg.frames])\n\n    def inputs(self, row, pass_name, index):\n        torch = self.torch\n        images, audio, key = self.media(row, pass_name, index)\n        media = ([{"type": "image"} for _ in images] + [{"type": "audio"}]) if images else []\n        def encode(question):\n            content = media + [{"type": "text", "text": "\\nQuestion: " + question + "\\nAnswer Yes or No only."}]\n            text = self.processor.apply_chat_template([{"role": "user", "content": content}],\n                        tokenize=False, add_generation_prompt=True, enable_thinking=False)\n            args = {"text": text, "return_tensors": "pt"}\n            if images:\n                args.update(images=images, audio=[audio], sampling_rate=16000)\n            if self.cfg.model == "qwen":\n                args["use_audio_in_video"] = False\n            encoded = self.processor(**args)\n            return {k: v.to(device="cuda", dtype=torch.float16 if v.is_floating_point() else v.dtype)\n                    for k, v in encoded.items() if isinstance(v, torch.Tensor)}\n        full, empty = encode(row["question"]), encode("")\n        a, b = full["input_ids"][0], empty["input_ids"][0]\n        n = min(len(a), len(b))\n        unequal = (a[:n] != b[:n]).nonzero()\n        cut = int(unequal[0]) if len(unequal) else n\n        if cut <= 0 or cut >= len(a):\n            raise RuntimeError("Cannot isolate question-independent media prefix")\n        if self.cfg.model == "qwen":\n            audio_lengths = full.get("feature_attention_mask")\n            audio_lengths = audio_lengths.sum(-1) if audio_lengths is not None else None\n            positions, _ = self.model.get_rope_index(\n                full["input_ids"], full.get("image_grid_thw"), None, full["attention_mask"],\n                False, audio_lengths, None)\n            full["position_ids"] = positions\n        else:\n            full["position_ids"] = torch.arange(len(a), device="cuda").unsqueeze(0)\n        key = digest([key, a[:cut].tolist()])\n        return full, cut, key\n\n    def split_inputs(self, full, cut, suffix=False):\n        length = full["input_ids"].shape[-1]\n        if not suffix:\n            result = dict(full)\n            for k in result:\n                if k not in ("input_ids", "attention_mask", "position_ids") and "token_type" not in k and "mm_type" not in k:\n                    continue\n                if k in result:\n                    result[k] = result[k][..., :cut]\n            return result\n        result = {"input_ids": full["input_ids"][:, cut:], "attention_mask": full["attention_mask"],\n                  "position_ids": full["position_ids"][..., cut:]}\n        for key, value in full.items():\n            if "token_type" in key or "mm_type" in key:\n                result[key] = value[..., cut:]\n        if self.cfg.model == "qwen":\n            result["cache_position"] = self.torch.arange(cut, length, device="cuda")\n        return result\n\n    def _forward(self, inputs, **kwargs):\n        with self.torch.inference_mode():\n            return self.model(**inputs, return_dict=True, output_hidden_states=True, **kwargs)\n\n    def extract(self, output):\n        import numpy as np\n        torch = self.torch\n        logits = output.logits[0, -1].float()\n        hidden = torch.stack([h[0, -1].float() for h in output.hidden_states])\n        if not torch.isfinite(logits).all() or not torch.isfinite(hidden).all():\n            raise FloatingPointError("fp16 overflow/nonfinite logits or hidden states; STOP")\n        yes = torch.logsumexp(logits[self.tokens["yes"]], 0)\n        no = torch.logsumexp(logits[self.tokens["no"]], 0)\n        margin = float(yes-no)\n        array = hidden.cpu().numpy()\n        if not np.isfinite(array.astype(np.float16)).all():\n            raise FloatingPointError("Feature shard would overflow fp16")\n        return {"margin": margin, "answer": int(margin > 0), "yes_logsumexp": float(yes),\n                "no_logsumexp": float(no), "greedy_token": int(logits.argmax())}, array, logits.cpu().numpy()\n\n    def score(self, row, pass_name, index, cached=None):\n        tick = time.monotonic()\n        mode = getattr(self.cfg, "scoring_mode", "verified_prefix")\n        if cached is None:\n            cached = mode == "verified_prefix"\n        if mode == "full_forward" and cached:\n            raise RuntimeError("Prefix reuse is disabled in full_forward mode")\n        full, cut, key = self.inputs(row, pass_name, index)\n        if not cached:\n            output = self._forward(full, use_cache=False)\n        else:\n            if self.prefix_key != key:\n                self.prefix = None\n                self.torch.cuda.empty_cache()\n                output = self._forward(self.split_inputs(full, cut), use_cache=True)\n                self.prefix = output.past_key_values\n                self.prefix_key = key\n                del output\n            # DynamicCache mutates; deepcopy must isolate both tensor storage and metadata.\n            cache = copy.deepcopy(self.prefix)\n            before = self.prefix.get_seq_length()\n            output = self._forward(self.split_inputs(full, cut, suffix=True), past_key_values=cache, use_cache=True)\n            if self.prefix.get_seq_length() != before:\n                raise RuntimeError("Prefix cache mutated across questions")\n        result, hidden, logits = self.extract(output)\n        del output\n        self.torch.cuda.synchronize()\n        result["peak_vram_bytes"] = self.torch.cuda.max_memory_allocated()\n        result["seconds"] = time.monotonic() - tick\n        result["scoring_mode"] = mode\n        return result, hidden, logits\n\n    def generation_check(self, row, index):\n        full, _, _ = self.inputs(row, "full", index)\n        # Reset model-level Qwen RoPE state; generate must compute its own media positions.\n        full.pop("position_ids", None)\n        if hasattr(self.model, "rope_deltas"):\n            self.model.rope_deltas = None\n        with self.torch.inference_mode():\n            output = self.model.generate(**full, max_new_tokens=1, do_sample=False,\n                                        return_dict_in_generate=True, output_scores=True)\n        scores = output.scores[0][0].float()\n        generated = int(output.sequences[0, -1])\n        return {"generated_token": generated, "generated_text": self.processor.tokenizer.decode([generated]),\n                "generated_answer": 1 if generated in self.tokens["yes"] else 0 if generated in self.tokens["no"] else None,\n                "margin": float(self.torch.logsumexp(scores[self.tokens["yes"]], 0) -\n                                self.torch.logsumexp(scores[self.tokens["no"]], 0))}\n\n\ndef record_scoring_amendment(cfg, evidence_file=None):\n    """Explicit pre-test fallback record; never relabel failed prefix parity as passed."""\n    if cfg.scoring_mode != "full_forward":\n        return\n    folder = Path(cfg.root) / cfg.model\n    target = folder / "scoring_amendment.json"\n    existing = read_json(target) if target.exists() else None\n    name = evidence_file or (existing["prefix_evidence_file"] if existing else "cache_diagnostic_v3.json")\n    if Path(name).name != name:\n        raise ValueError("Failure evidence must be a filename inside the model study folder")\n    evidence_path = folder / name\n    evidence = read_json(evidence_path)\n    if evidence.get("status") != "comparison_complete" or not (\n            evidence["logits_max_abs_diff"] > .02 or evidence["hidden_max_abs_diff"] > .02\n            or len(set(evidence["answers"])) != 1):\n        raise RuntimeError("Full-forward fallback requires the saved failed prefix diagnostic")\n    payload = {"scoring_mode": "full_forward", "prefix_cache_status": "failed_gate_disabled",\n        "model": MODEL_IDS[cfg.model], "revision": cfg.revision, "frames": cfg.frames,\n        "reason": "Use fresh full forward as reference after prefix parity failed; no tolerance relaxation",\n        "prefix_evidence_file": evidence_path.name, "prefix_evidence_sha256": file_hash(evidence_path),\n        "prefix_evidence": evidence, "code_sha256": file_hash(__file__),\n        "formal_checks": "full-forward repeatability, generation agreement, finiteness and VRAM",\n        "statistics": "unchanged; separate formal smoke and pre-test analysis freeze still required"}\n    if existing is not None:\n        # Preserve the code hash at the time of the original mode decision. Current\n        # code is independently hashed in every run identity and smoke artifact.\n        changed = any(existing.get(k) != v for k, v in payload.items() if k != "code_sha256")\n        if changed:\n            raise RuntimeError("Scoring amendment changed; do not reuse an existing run identity")\n        return\n    atomic_json(target, payload)\n\n\ndef use_full_forward(scorer, cfg, evidence_file=None):\n    """Upgrade a retained scorer\'s dispatch without allocating/reloading model weights."""\n    from types import MethodType\n    updated = Config(**{**dataclasses.asdict(cfg), "scoring_mode": "full_forward"})\n    for key in ("model", "revision", "frames", "max_side"):\n        if getattr(scorer.cfg, key) != getattr(updated, key):\n            raise RuntimeError("Retained model/config mismatch: " + key)\n    record_scoring_amendment(updated, evidence_file=evidence_file)\n    scorer.cfg = updated\n    scorer.score = MethodType(OmniScorer.score, scorer)\n    scorer.env = dict(scorer.env, scoring_mode="full_forward")\n    scorer.prefix = None\n    scorer.prefix_key = None\n    return updated\n\n\ndef recover_full_forward():\n    """Recover the retained model and actual prefix-failure evidence from IPython."""\n    import sys\n    import gc\n    old_tb = getattr(sys, "last_traceback", None)\n    current, retained, entry = old_tb, None, None\n    while current is not None:\n        frame = current.tb_frame\n        if frame.f_code.co_name == "parity_checks" and Path(frame.f_code.co_filename).name == "avh_common.py":\n            retained = frame.f_locals.get("scorer")\n            observed = frame.f_locals.get("entry")\n            entry = dict(observed) if observed is not None else None\n        current = current.tb_next\n    if retained is None or entry is None:\n        raise RuntimeError("Failed smoke model/evidence unavailable; send this output before reloading any model")\n    if entry.get("comparison_kind", "prefix_parity") != "prefix_parity":\n        raise RuntimeError("Latest failure is not prefix parity; full-forward repeatability must not be bypassed")\n    cfg = retained.cfg\n    evidence = {"status": "comparison_complete", "kind": "captured_original_prefix_failure",\n        "model": MODEL_IDS[cfg.model], "revision": cfg.revision, "n_qa_diagnostic": 1,\n        "logits_max_abs_diff": entry["logits_max_abs_diff"],\n        "hidden_max_abs_diff": entry["hidden_max_abs_diff"],\n        "answers": [entry["fresh_answer"], entry["cached_answer"]], "failed_check": entry}\n    atomic_json(Path(cfg.root) / cfg.model / "cache_parity_failure.json", evidence)\n    cfg = use_full_forward(retained, cfg, evidence_file="cache_parity_failure.json")\n    if old_tb is not None:\n        traceback.clear_frames(old_tb)\n    for name in ("last_traceback", "last_value", "last_exc"):\n        setattr(sys, name, None)\n    del old_tb, current, frame\n    gc.collect()\n    if hasattr(retained, "torch"):\n        retained.torch.cuda.empty_cache()\n    return retained, cfg\n\n\ndef study_signature(cfg, manifest, media_index, root, environment):\n    return digest({"model": cfg.model, "revision": cfg.revision, "frames": cfg.frames,\n                   "max_side": cfg.max_side, "seed": cfg.seed, "k": cfg.k,\n                   "workers": cfg.workers, "manifest": digest(manifest), "media": digest(media_index),\n                   "scoring_mode": cfg.scoring_mode,\n                   "scoring_amendment": file_hash(Path(root) / cfg.model / "scoring_amendment.json")\n                       if cfg.scoring_mode == "full_forward" else None,\n                   "terms": file_hash(Path(root) / "terms_review.json"),\n                   "preregistration": file_hash(Path(root) / "PREREGISTRATION.md"),\n                   "code": file_hash(__file__), "environment": environment})\n\n\ndef smoke_test(scorer, cfg, manifest, index, owner_release):\n    require_go(owner_release)\n    if getattr(getattr(scorer, "cfg", None), "scoring_mode", "verified_prefix") != cfg.scoring_mode:\n        raise RuntimeError("Scorer mode differs from smoke configuration")\n    check_terms(cfg.root, cfg.model)\n    verify_media(index)\n    # Calibration only. Test data remain unseen during all engineering choices.\n    items = sorted((r for r in manifest["rows"] if r["split"] == "calibration" and r["task"] != TASKS[2]),\n                   key=lambda r: digest(["smoke", r["id"]]))[:20]\n    if len(items) != 20 or len({r["video_id"] for r in items}) < 2:\n        raise RuntimeError("Need 20 calibration questions spanning multiple media items")\n    report = parity_checks(scorer, items, index, all_passes=True)\n    result = {"passed": True, "configuration": dataclasses.asdict(cfg), "environment": scorer.env,\n              "comparison_kind": "prefix_parity" if cfg.scoring_mode == "verified_prefix" else "full_forward_repeatability",\n              "prefix_cache_verified": cfg.scoring_mode == "verified_prefix",\n              "manifest_sha256": digest(manifest), "media_sha256": digest(index),\n              "code_sha256": file_hash(__file__), "tokens": scorer.tokens, "checks": report}\n    atomic_json(Path(cfg.root) / cfg.model / f"smoke_f{cfg.frames}.json", result)\n    return result\n\n\ndef parity_checks(scorer, items, index, all_passes=False):\n    """Shared diagnostics: identical parity, generation, finiteness and VRAM checks."""\n    report = []\n    use_prefix = getattr(getattr(scorer, "cfg", None), "scoring_mode", "verified_prefix") == "verified_prefix"\n    with Watchdog(600) as dog:\n        for i, row in enumerate(items):\n            # Every pass can change media length/positioning. Check all pass paths.\n            for pass_name in (PASSES if all_passes and i == 0 else ("full",)):\n                fresh, fh, fl = scorer.score(row, pass_name, index, cached=False)\n                cached, ch, cl = scorer.score(row, pass_name, index, cached=use_prefix)\n                import numpy as np\n                ld, hd = float(np.max(np.abs(fl-cl))), float(np.max(np.abs(fh-ch)))\n                entry = {"id": row["id"], "pass": pass_name, "logits_max_abs_diff": ld,\n                         "hidden_max_abs_diff": hd, "fresh_answer": fresh["answer"],\n                         "cached_answer" if use_prefix else "repeat_answer": cached["answer"],\n                         "comparison_kind": "prefix_parity" if use_prefix else "full_forward_repeatability",\n                         "fresh_seconds": fresh.get("seconds"),\n                         "peak_vram_bytes": cached["peak_vram_bytes"]}\n                if i < 10 and pass_name == "full":\n                    gen = scorer.generation_check(row, index)\n                    entry["generation"] = gen\n                    if gen["generated_answer"] != fresh["answer"]:\n                        raise RuntimeError("generate/Yes-No scoring disagreement; inspect prompt/token mapping")\n                report.append(entry)\n                if ld > .02 or hd > .02 or fresh["answer"] != cached["answer"]:\n                    name = "Cache parity" if use_prefix else "Full-forward repeatability"\n                    raise RuntimeError(name + " failed: " + json.dumps(entry))\n                dog.touch()\n    return report\n\n\ndef pre_smoke(cfg, raw_qa, media_dir, scorer=None):\n    """Early private engineering diagnostic only; no terms, lineage, manifest or result gate.\n\n    Caller notebook still requires owner GO. These raw items can later be in test;\n    do not use their answers to tune the research protocol or claim a result.\n    """\n    import numpy as np\n    record_scoring_amendment(cfg)\n    pool = [dict(r) for r in raw_qa if r["task"] in TASKS[:2]]\n    videos = sorted({str(r["video_id"]) for r in pool})\n    if len(pool) < 20 or len(videos) < 3:\n        raise RuntimeError("Early smoke needs 20 raw QA rows from at least three videos")\n    rng = np.random.default_rng(cfg.seed)\n    order = rng.permutation(len(pool)).tolist()\n    selected, covered = [], set()\n    for i in order:\n        if str(pool[i]["video_id"]) not in covered:\n            selected.append(i)\n            covered.add(str(pool[i]["video_id"]))\n        if len(covered) == 3:\n            break\n    selected.extend(i for i in order if i not in selected)\n    items = [pool[i] for i in selected[:20]]\n    for row in items:\n        row.update(video_id=str(row["video_id"]), question=row["text"],\n                   id=digest(["pre-smoke", row["video_id"], row["task"], row["text"]])[:24])\n    diagnostic_root = Path(cfg.root) / cfg.model / "pre_smoke_media"\n    index = prepare_media(diagnostic_root, media_dir, {"rows": items}, cfg.frames, cfg.max_side)\n    verify_media(index)\n    started = time.monotonic()\n    timeout = 1800 if cfg.model == "qwen" else 3600\n    with Watchdog(timeout, deadline=started+1800 if cfg.model == "qwen" else None) as dog:\n        reused_model = scorer is not None\n        if scorer is None:\n            scorer = OmniScorer(cfg)\n        else:\n            for key in ("model", "revision", "frames", "max_side"):\n                if getattr(scorer.cfg, key) != getattr(cfg, key):\n                    raise RuntimeError("Retained model/config mismatch: " + key)\n            scorer.cfg = cfg\n            scorer.env = dict(scorer.env, scoring_mode=cfg.scoring_mode)\n            scorer.prefix = None\n            scorer.prefix_key = None\n            if hasattr(scorer, "torch"):\n                scorer.torch.cuda.reset_peak_memory_stats()\n        dog.touch()\n        checks = parity_checks(scorer, items, index)\n    result = {"kind": "engineering_diagnostic_only_not_a_result", "passed": True,\n        "eligible_for_bulk_gate": False, "configuration": dataclasses.asdict(cfg),\n        "reused_loaded_model": reused_model,\n        "comparison_kind": "prefix_parity" if cfg.scoring_mode == "verified_prefix" else "full_forward_repeatability",\n        "prefix_cache_verified": cfg.scoring_mode == "verified_prefix",\n        "seconds_load_and_checks": time.monotonic()-started, "n_qa": len(items),\n        "n_videos": len({r["video_id"] for r in items}), "selection_seed": cfg.seed,\n        "raw_selection_sha256": digest(items), "environment": scorer.env,\n        "tokens": scorer.tokens, "checks": checks, "media_sha256": digest(index)}\n    times = [r["fresh_seconds"] for r in checks if r["pass"] == "full" and r["fresh_seconds"] is not None]\n    if len(times) == len(items):\n        result["observed_full_pass_seconds"] = cluster_ci(\n            times, [r["video_id"] for r in items], lambda a: float(a.mean()))\n    atomic_json(Path(cfg.root) / cfg.model / "pre_smoke.json", result)\n    return scorer, result\n\n\ndef run_stage(scorer, cfg, manifest, index, owner_release, splits=("calibration", "validation")):\n    require_go(owner_release)\n    if getattr(getattr(scorer, "cfg", None), "scoring_mode", "verified_prefix") != cfg.scoring_mode:\n        raise RuntimeError("Scorer mode differs from run configuration")\n    check_terms(cfg.root, cfg.model)\n    verify_media(index)\n    smoke = read_json(Path(cfg.root) / cfg.model / f"smoke_f{cfg.frames}.json")\n    if (not smoke.get("passed") or smoke["manifest_sha256"] != digest(manifest)\n            or smoke["media_sha256"] != digest(index) or smoke["code_sha256"] != file_hash(__file__)\n            or smoke["environment"] != scorer.env):\n        raise RuntimeError("Matching smoke gate missing/stale")\n    for key in ("revision", "frames", "max_side", "seed", "model", "scoring_mode"):\n        if smoke["configuration"][key] != getattr(cfg, key):\n            raise RuntimeError("Smoke configuration mismatch")\n    if "test" in splits:\n        freeze = read_json(Path(cfg.root) / cfg.model / "analysis_freeze.json")\n        if (freeze.get("status") != "frozen_before_test" or freeze["manifest_sha256"] != digest(manifest)\n                or freeze["signature"] != study_signature(cfg, manifest, index, cfg.root, scorer.env)):\n            raise RuntimeError("Freeze CPU validation choices before opening test inference")\n    signature = study_signature(cfg, manifest, index, cfg.root, scorer.env)\n    directory = Path(cfg.root) / cfg.model / f"f{cfg.frames}" / cfg.stage\n    items = [r for r in manifest["rows"] if r["split"] in splits\n             and (r["task"] != TASKS[2] or cfg.stage in ("full", "no_media"))\n             and int(digest(r["id"])[:16], 16) % cfg.workers == cfg.worker_index]\n    # Group same media/donor prefixes for encode-once reuse across questions.\n    items.sort(key=lambda r: (r["video_id"], r["task"], tuple(r["donors"]), r["id"]))\n    worker = f"partition-{cfg.worker_index}-of-{cfg.workers}"\n    with Journal(directory, worker, signature) as journal, Watchdog() as dog:\n        start, added = time.monotonic(), 0\n        for row in items:\n            if row["id"] in journal.done:\n                continue\n            try:\n                tick = time.monotonic()\n                result, hidden, _ = scorer.score(row, cfg.stage, index)\n                result.update({k: row[k] for k in ("id", "video_id", "source_group", "split", "task", "label", "object")})\n                result.update({"pass": cfg.stage, "seconds": time.monotonic()-tick})\n                journal.append(result, hidden if cfg.stage == "full" else None)\n                dog.touch()\n                added += 1\n                if added % 100 == 0:\n                    print(json.dumps({"new_rows": added, "seconds_per_question": (time.monotonic()-start)/added,\n                                      "peak_vram_bytes": result["peak_vram_bytes"]}), flush=True)\n                if added % 200 == 0:\n                    journal.compact()\n            except BaseException:\n                traceback.print_exc()\n                raise\n        journal.compact()\n    return {"new_rows": added, "eligible": len(items), "stage": cfg.stage, "signature": signature}\n\n\ndef cluster_ci(values, groups, statistic, resamples=2000, seed=SEED):\n    """Paired resampling unit is connected source component; undefined draws discarded explicitly."""\n    import numpy as np\n    values = np.asarray(values)\n    groups = np.asarray(groups)\n    unique = np.unique(groups)\n    if not len(values):\n        return {"estimate": None, "ci95": None, "n": 0, "n_clusters": 0, "valid_resamples": 0}\n    estimate = statistic(values)\n    rng = np.random.default_rng(seed)\n    buckets = {g: np.flatnonzero(groups == g) for g in unique}\n    draws = []\n    if len(unique) >= 2:\n        for _ in range(resamples):\n            indices = np.concatenate([buckets[g] for g in rng.choice(unique, len(unique), replace=True)])\n            value = statistic(values[indices])\n            if value is not None and np.isfinite(value):\n                draws.append(value)\n    return {"estimate": float(estimate) if estimate is not None and np.isfinite(estimate) else None,\n            "ci95": np.quantile(draws, [.025, .975]).tolist() if len(draws) >= .9 * resamples else None,\n            "n": len(values), "n_clusters": len(unique), "valid_resamples": len(draws),\n            "requested_resamples": resamples, "seed": seed}\n\n\ndef rates(labels, answers, groups):\n    import numpy as np\n    y, p, g = np.asarray(labels), np.asarray(answers), np.asarray(groups)\n    return {"accuracy": cluster_ci((y == p).astype(float), g, lambda x: float(x.mean())),\n            "false_yes": cluster_ci(p[y == 0].astype(float), g[y == 0], lambda x: float(x.mean())),\n            "recall": cluster_ci(p[y == 1].astype(float), g[y == 1], lambda x: float(x.mean())),\n            "n_false_yes": int(((y == 0) & (p == 1)).sum()), "n": len(y),\n            "n_negative": int((y == 0).sum()), "n_positive": int((y == 1).sum()),\n            "n_correct": int((y == p).sum()), "n_true_yes": int(((y == 1) & (p == 1)).sum()),\n            "power": "underpowered" if ((y == 0) & (p == 1)).sum() < 40 else "at_least_40_false_yes"}\n\n\ndef load_completed(root, cfg, manifest, index, environment, split):\n    """Strict completeness: partial runs never become a metrics result."""\n    import numpy as np\n    signature = study_signature(cfg, manifest, index, root, environment)\n    excluded = check_audit(root, manifest)\n    freeze_path = Path(root) / cfg.model / "analysis_freeze.json"\n    if (split == "test" and manifest.get("test_scoring_policy") == "sealed_precompute"\n            and not freeze_path.exists()):\n        raise RuntimeError("Sealed test predictions cannot be opened before analysis freeze")\n    if split == "test" and manifest.get("test_scoring_policy") == "sealed_precompute":\n        sealed_freeze = read_json(freeze_path)\n        if (sealed_freeze.get("status") != "frozen_before_test_evaluation"\n                or sealed_freeze.get("signature") != signature\n                or file_hash(Path(root) / cfg.model / "analysis_bundle.joblib") != sealed_freeze.get("bundle_sha256")):\n            raise RuntimeError("Sealed test analysis freeze is missing, invalid or changed")\n    if freeze_path.exists() and read_json(freeze_path)["audit_sha256"] != file_hash(Path(root) / "audit.csv"):\n        raise RuntimeError("Audit changed after analysis freeze")\n    wanted = {r["id"]: r for r in manifest["rows"] if r["split"] == split and r["id"] not in excluded}\n    stores = {}\n    stages = ("full", *(f"cf{k}" for k in range(cfg.k)), "blank", "no_media")\n    for stage in stages:\n        directory = Path(root) / cfg.model / f"f{cfg.frames}" / stage\n        rows = {}\n        for path in directory.glob("*.jsonl"):\n            for line in path.read_text(encoding="utf-8").splitlines():\n                row = json.loads(line)\n                if row["signature"] != signature:\n                    raise RuntimeError("Mixed run/environment signature")\n                if row["id"] in rows:\n                    raise RuntimeError("Duplicate work across partitions")\n                rows[row["id"]] = row\n        required = {k for k, r in wanted.items() if r["task"] != TASKS[2] or stage in ("full", "no_media")}\n        missing = required - rows.keys()\n        if missing:\n            raise RuntimeError(f"Incomplete {stage}/{split}: {len(missing)} missing questions; no extrapolation")\n        stores[stage] = rows\n    ids = sorted(wanted)\n    data, feature_hashes = [], {}\n    for key in ids:\n        row = dict(wanted[key])\n        row["margin"] = stores["full"][key]["margin"]\n        row["no_media_margin"] = stores["no_media"][key]["margin"]\n        row["contrast"] = (row["margin"] - np.mean([stores[f"cf{k}"][key]["margin"] for k in range(cfg.k)])\n                           if row["task"] != TASKS[2] else None)\n        row["blank_contrast"] = (row["margin"] - stores["blank"][key]["margin"] if row["task"] != TASKS[2] else None)\n        frow = stores["full"][key]\n        directory = Path(root) / cfg.model / f"f{cfg.frames}" / "full"\n        feature = directory / frow["feature"]\n        if feature not in feature_hashes:\n            feature_hashes[feature] = file_hash(feature)\n        if feature_hashes[feature] != frow["feature_sha256"]:\n            raise RuntimeError("Feature hash mismatch")\n        if "feature_index" in frow:\n            row["hidden"] = np.load(feature, mmap_mode="r", allow_pickle=False)[frow["feature_index"]].astype(np.float32)\n        else:\n            with np.load(feature, allow_pickle=False) as archive:\n                row["hidden"] = archive["hidden"].astype(np.float32)\n        data.append(row)\n    return data\n\n\ndef safe_auc(y, score):\n    import numpy as np\n    from sklearn.metrics import roc_auc_score\n    return float(roc_auc_score(y, score)) if len(np.unique(y)) == 2 else None\n\n\ndef auc_ci(y, score, groups):\n    import numpy as np\n    return cluster_ci(np.column_stack([y, score]), groups, lambda a: safe_auc(a[:, 0], a[:, 1]))\n\n\ndef object_rates(calibration, rows, leave_group_out=False):\n    """Smoothed model-Yes propensity, never held-out ground-truth object prevalence."""\n    import numpy as np\n    result = []\n    for row in rows:\n        pool = [r for r in calibration if r["task"] == row["task"] and r["object"] == row["object"]\n                and (not leave_group_out or r["source_group"] != row["source_group"])]\n        result.append((1 + sum(r["margin"] > 0 for r in pool)) / (2 + len(pool)))\n    return np.asarray(result)\n\n\ndef baseline_matrix(calibration, rows, training=False):\n    import numpy as np\n    return np.column_stack([[r["margin"] for r in rows], [r["contrast"] for r in rows],\n                            [r["no_media_margin"] for r in rows],\n                            object_rates(calibration, rows, leave_group_out=training)])\n\n\nclass Residualizer:\n    """Unlabeled calibration centering + ridge regression on answer margin; frozen transforms."""\n    def fit(self, rows, layer, direction=None):\n        import numpy as np\n        self.layer, self.direction = layer, direction\n        self.centers = {t: np.stack([r["hidden"][layer] for r in rows if r["task"] == t]).mean(0)\n                        for t in {r["task"] for r in rows}}\n        x = np.stack([r["hidden"][layer] - self.centers[r["task"]] for r in rows])\n        m = np.column_stack([np.ones(len(rows)), [r["margin"] for r in rows]])\n        self.beta = np.linalg.solve(m.T @ m + np.diag([1e-8, 1e-4]), m.T @ x)\n        return self\n\n    def transform(self, rows):\n        import numpy as np\n        x = np.stack([r["hidden"][self.layer] - self.centers[r["task"]] for r in rows])\n        m = np.column_stack([np.ones(len(rows)), [r["margin"] for r in rows]])\n        x = x - m @ self.beta\n        if self.direction is not None:\n            d = self.direction / max(np.linalg.norm(self.direction), 1e-12)\n            if len(d) != x.shape[1]:\n                raise ValueError("Unembedding and residual dimension mismatch")\n            x = x - np.outer(x @ d, d)\n        return x\n\n\ndef logistic(x, y, c):\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler\n    from sklearn.linear_model import LogisticRegression\n    return make_pipeline(StandardScaler(), LogisticRegression(C=c, solver="liblinear", max_iter=2000,\n                                                              random_state=SEED)).fit(x, y)\n\n\ndef select_logistic(x, y, vx, vy):\n    options = []\n    for c in (.01, .1, 1.):\n        model = logistic(x, y, c)\n        auc = safe_auc(vy, model.predict_proba(vx)[:, 1])\n        if auc is not None:\n            options.append((auc, -c, model))\n    if not options:\n        raise ValueError("Both classes required in calibration and validation")\n    best = max(options, key=lambda a: a[:2])\n    return best[2], best[0]\n\n\ndef fit_task_probe(calibration, validation, task, unembedding):\n    import numpy as np\n    cal = [r for r in calibration if r["task"] == task and r["margin"] > 0]\n    val = [r for r in validation if r["task"] == task and r["margin"] > 0]\n    counts = {"calibration_yes": len(cal), "validation_yes": len(val),\n              "calibration_false_yes": sum(not r["label"] for r in cal),\n              "validation_false_yes": sum(not r["label"] for r in val)}\n    if not cal or not val or len({r["label"] for r in cal}) < 2 or len({r["label"] for r in val}) < 2:\n        return {"status": "underpowered_missing_class", "counts": counts}\n    y, vy = np.array([1-r["label"] for r in cal]), np.array([1-r["label"] for r in val])\n    bx, bv = baseline_matrix(calibration, cal, True), baseline_matrix(calibration, val)\n    base, _ = select_logistic(bx, y, bv, vy)\n    # Use all unlabeled calibration rows for task centering/margin residualization.\n    choices = []\n    for layer in range(cal[0]["hidden"].shape[0]):\n        transform = Residualizer().fit(calibration, layer)\n        x, vx = transform.transform(cal), transform.transform(val)\n        model, score = select_logistic(np.column_stack([bx, x]), y, np.column_stack([bv, vx]), vy)\n        choices.append((score, -layer, transform, model))\n    score, neg_layer, transform, augmented = max(choices, key=lambda a: a[:2])\n    layer = -neg_layer\n    x, vx = transform.transform(cal), transform.transform(val)\n    standalone, _ = select_logistic(x, y, vx, vy)\n    # Same selected layer/C and same preprocessing for the permutation control.\n    shuffled_y = np.random.default_rng(SEED).permutation(y)\n    shuffled = logistic(np.column_stack([bx, x]), shuffled_y,\n                        augmented.named_steps["logisticregression"].C)\n    projected_transform = Residualizer().fit(calibration, layer, unembedding)\n    px, pv = projected_transform.transform(cal), projected_transform.transform(val)\n    projected, _ = select_logistic(np.column_stack([bx, px]), y, np.column_stack([bv, pv]), vy)\n    return {"status": "fitted", "counts": counts, "layer": layer,\n            "baseline": base, "augmented": augmented, "standalone": standalone,\n            "shuffled": shuffled, "transform": transform, "projected": projected,\n            "projected_transform": projected_transform,\n            "validation_margin_edges": np.quantile([r["margin"] for r in val], [.2, .4, .6, .8]).tolist()}\n\n\ndef threshold_select(rows, field, risk=False):\n    """Max validation false-yes reduction at <=.02 point recall cost, ties favor coverage."""\n    import numpy as np\n    y = np.array([r["label"] for r in rows])\n    original = np.array([r["margin"] > 0 for r in rows])\n    score = np.asarray([r[field] for r in rows], dtype=float)\n    if not (np.any(y == 0) and np.any(y == 1)) or not np.isfinite(score).all():\n        raise ValueError("Threshold selection needs finite scores and positive/negative labels")\n    values = np.unique(score)\n    thresholds = np.r_[values[0]-1, values, values[-1]+1]\n    original_recall = original[y == 1].mean()\n    candidates = []\n    for threshold in thresholds:\n        accepted = original & ((score <= threshold) if risk else (score >= threshold))\n        if original_recall - accepted[y == 1].mean() <= .02 + 1e-12:\n            candidates.append((accepted[y == 0].mean(), -accepted.mean(), float(threshold)))\n    return min(candidates) [2]\n\n\ndef probe_scores(bundle, calibration, rows):\n    import numpy as np\n    if bundle["status"] != "fitted":\n        return None\n    b = baseline_matrix(calibration, rows)\n    x = bundle["transform"].transform(rows)\n    p = bundle["projected_transform"].transform(rows)\n    return {"baseline": bundle["baseline"].predict_proba(b)[:, 1],\n            "augmented": bundle["augmented"].predict_proba(np.column_stack([b, x]))[:, 1],\n            "probe_only": bundle["standalone"].predict_proba(x)[:, 1],\n            "shuffled": bundle["shuffled"].predict_proba(np.column_stack([b, x]))[:, 1],\n            "projected": bundle["projected"].predict_proba(np.column_stack([b, p]))[:, 1]}\n\n\ndef freeze_analysis(root, cfg, manifest, index, environment, unembedding):\n    """CPU only. Fits calibration, selects validation, saves immutable choices before test."""\n    import numpy as np\n    import joblib\n    check_audit(root, manifest, require_complete=True)\n    calibration = load_completed(root, cfg, manifest, index, environment, "calibration")\n    validation = load_completed(root, cfg, manifest, index, environment, "validation")\n    signature = study_signature(cfg, manifest, index, root, environment)\n    out = Path(root) / cfg.model\n    freeze_path = out / "analysis_freeze.json"\n    if freeze_path.exists():\n        existing = read_json(freeze_path)\n        if existing["signature"] != signature:\n            raise RuntimeError("Frozen analysis signature mismatch")\n        if file_hash(out / "analysis_bundle.joblib") != existing["bundle_sha256"]:\n            raise RuntimeError("Analysis bundle changed")\n        if manifest.get("test_scoring_policy") == "sealed_precompute":\n            marker = out / "FROZEN.json"\n            if marker.exists() and read_json(marker) != existing:\n                raise RuntimeError("Sealed analysis freeze markers disagree")\n            if not marker.exists():\n                atomic_json(marker, existing)\n        return existing\n    bundles, thresholds, summary = {}, {}, {}\n    for task in TASKS[:2]:\n        val = [dict(r) for r in validation if r["task"] == task]\n        if not val:\n            raise RuntimeError("No validation rows for a primary task")\n        bundle = fit_task_probe(calibration, validation, task, np.asarray(unembedding))\n        bundles[task] = bundle\n        thresholds[task] = {field: threshold_select(val, field) for field in ("contrast", "blank_contrast")}\n        summary[task] = {k: bundle[k] for k in ("status", "counts")}\n        if bundle["status"] == "fitted":\n            yes = [r for r in val if r["margin"] > 0]\n            scores = probe_scores(bundle, calibration, yes)\n            mapping = {r["id"]: float(s) for r, s in zip(yes, scores["augmented"])}\n            for row in val:\n                row["probe_risk"] = mapping.get(row["id"], 0.)\n            thresholds[task]["probe_risk"] = threshold_select(val, "probe_risk", risk=True)\n            summary[task]["selected_layer"] = bundle["layer"]\n    path = out / "analysis_bundle.joblib"\n    # Bundle is local trusted analysis state; never load a third-party pickle/joblib file.\n    joblib.dump({"bundles": bundles, "calibration": calibration, "thresholds": thresholds}, path)\n    freeze = {"status": "frozen_before_test_evaluation" if manifest.get("test_scoring_policy") == "sealed_precompute" else "frozen_before_test",\n              "signature": signature, "manifest_sha256": digest(manifest),\n              "audit_sha256": file_hash(Path(root) / "audit.csv"),\n              "bundle_sha256": file_hash(path), "thresholds": thresholds, "probe_choices": summary,\n              "environment": environment, "unembedding_sha256": digest(np.asarray(unembedding).tolist())}\n    atomic_json(freeze_path, freeze)\n    if manifest.get("test_scoring_policy") == "sealed_precompute":\n        atomic_json(out / "FROZEN.json", freeze)\n    return freeze\n\n\ndef evaluate_policy(rows, accepted, resamples=2000):\n    import numpy as np\n    y = np.array([r["label"] for r in rows])\n    original = np.array([r["margin"] > 0 for r in rows])\n    g = np.array([r["source_group"] for r in rows])\n    accepted = np.asarray(accepted, dtype=bool)\n    false_delta = cluster_ci((original[y == 0].astype(float)-accepted[y == 0]).astype(float), g[y == 0],\n                             lambda a: float(a.mean()), resamples)\n    recall_drop = cluster_ci((original[y == 1].astype(float)-accepted[y == 1]).astype(float), g[y == 1],\n                             lambda a: float(a.mean()), resamples)\n    supported = (false_delta["ci95"] is not None and false_delta["ci95"][0] > 0\n                 and recall_drop["estimate"] is not None and recall_drop["estimate"] <= .02 + 1e-12)\n    return {"false_yes_before": cluster_ci(original[y == 0].astype(float), g[y == 0], lambda a: float(a.mean()), resamples),\n            "false_yes_after": cluster_ci(accepted[y == 0].astype(float), g[y == 0], lambda a: float(a.mean()), resamples),\n            "false_yes_reduction": false_delta, "recall_drop": recall_drop,\n            "recall_before": cluster_ci(original[y == 1].astype(float), g[y == 1], lambda a: float(a.mean()), resamples),\n            "recall_after": cluster_ci(accepted[y == 1].astype(float), g[y == 1], lambda a: float(a.mean()), resamples),\n            "share_answered": cluster_ci((~original | accepted).astype(float), g, lambda a: float(a.mean()), resamples),\n            "verdict": "ship_demo_flag" if supported else "did_not_ship",\n            "n": len(rows), "n_false_yes_before": int(((y == 0) & original).sum()),\n            "power": "underpowered" if ((y == 0) & original).sum() < 40 else "at_least_40_false_yes"}\n\n\ndef evaluate_test(root, cfg, manifest, index, environment, owner_release):\n    require_go(owner_release)\n    import numpy as np\n    import joblib\n    freeze = read_json(Path(root) / cfg.model / "analysis_freeze.json")\n    if freeze["signature"] != study_signature(cfg, manifest, index, root, environment):\n        raise RuntimeError("Freeze does not match run")\n    path = Path(root) / cfg.model / "analysis_bundle.joblib"\n    if file_hash(path) != freeze["bundle_sha256"]:\n        raise RuntimeError("Bundle hash mismatch")\n    bundle = joblib.load(path)\n    test = load_completed(root, cfg, manifest, index, environment, "test")\n    terms = check_terms(root, cfg.model)\n    metrics = {"study": "CounterPrior v2", "date": "2026-11", "model": MODEL_IDS[cfg.model],\n               "scoring_mode": cfg.scoring_mode,\n               "prefix_cache_verified": cfg.scoring_mode == "verified_prefix",\n               "revision": cfg.revision, "signature": freeze["signature"], "tasks": {},\n               "method": "AVCD-style queried-modality contrast; not published attentive-mask AVCD",\n               "uncertainty": "2000 paired grouping-unit bootstrap draws; conditional on fixed donor pool; see lineage limitations",\n               "limitations": manifest.get("limitations", []), "object_rule": manifest.get("object_rule", OBJECT_RULE),\n               "dataset_terms_decision": terms["dataset"]["decision"],\n               "public_avhbench_media_allowed": False,\n               "publication_policy": "Aggregate numbers and our own text only; no AVHBench audio, video or frames" if\n                   terms["dataset"]["decision"] == "no_licence_stated_private_research_eval" else\n                   "No public media authorization granted by this evaluation; separate permission required",\n               "audit_sha256": freeze["audit_sha256"],\n               "preregistration_sha256": file_hash(Path(root) / "PREREGISTRATION.md")}\n    for task in TASKS:\n        rows = [r for r in test if r["task"] == task]\n        if not rows:\n            metrics["tasks"][task] = {"status": "not_available"}\n            continue\n        y, g = np.array([r["label"] for r in rows]), [r["source_group"] for r in rows]\n        original = np.array([r["margin"] > 0 for r in rows])\n        out = {"full": rates(y, original, g),\n               "no_media": rates(y, [r["no_media_margin"] > 0 for r in rows], g)}\n        no_media = np.array([r["no_media_margin"] > 0 for r in rows])\n        out["full_minus_no_media"] = {\n            "accuracy": cluster_ci((original == y).astype(float)-(no_media == y), g, lambda a: float(a.mean())),\n            "false_yes": cluster_ci(original[y == 0].astype(float)-no_media[y == 0], np.array(g)[y == 0], lambda a: float(a.mean())),\n            "recall": cluster_ci(original[y == 1].astype(float)-no_media[y == 1], np.array(g)[y == 1], lambda a: float(a.mean()))}\n        if task == TASKS[2]:\n            metrics["tasks"][task] = out\n            continue\n        thresholds = bundle["thresholds"][task]\n        for field in ("contrast", "blank_contrast"):\n            accepted = original & (np.array([r[field] for r in rows]) >= thresholds[field])\n            out[field] = evaluate_policy(rows, accepted)\n            out[field]["validation_threshold"] = thresholds[field]\n        pb = bundle["bundles"][task]\n        yes = [r for r in rows if r["margin"] > 0]\n        if pb["status"] != "fitted" or not yes:\n            out["probe"] = {"status": "underpowered_missing_class", "calibration_validation": pb["counts"]}\n        else:\n            scores = probe_scores(pb, bundle["calibration"], yes)\n            target = np.array([1-r["label"] for r in yes])\n            group = [r["source_group"] for r in yes]\n            probe = {"selected_layer": pb["layer"], "n_yes": len(yes), "n_false_yes": int(target.sum()),\n                     "power": "underpowered" if target.sum() < 40 else "at_least_40_false_yes",\n                     "auroc": {k: auc_ci(target, v, group) for k, v in scores.items()}}\n            paired = np.column_stack([target, scores["augmented"], scores["baseline"]])\n            def delta(a):\n                a1, a0 = safe_auc(a[:, 0], a[:, 1]), safe_auc(a[:, 0], a[:, 2])\n                return a1-a0 if a1 is not None and a0 is not None else None\n            probe["delta_auroc_augmented_over_baseline"] = cluster_ci(paired, group, delta)\n            bins = np.digitize([r["margin"] for r in yes], pb["validation_margin_edges"])\n            probe["within_margin_quintiles"] = {}\n            for q in range(5):\n                mask = bins == q\n                probe["within_margin_quintiles"][str(q+1)] = {\n                    "delta_auroc": cluster_ci(paired[mask], np.array(group)[mask], delta),\n                    "n_false_yes": int(target[mask].sum()),\n                    "power": "underpowered" if target[mask].sum() < 40 else "at_least_40_false_yes"}\n            risk = {r["id"]: s for r, s in zip(yes, scores["augmented"])}\n            accepted = np.array([r["margin"] > 0 and risk[r["id"]] <= thresholds["probe_risk"] for r in rows])\n            probe["policy"] = evaluate_policy(rows, accepted)\n            # Paired, measured difference versus contrast alone, not just each versus no fix.\n            contrast_accepted = original & (np.array([r["contrast"] for r in rows]) >= thresholds["contrast"])\n            probe["false_yes_reduction_vs_contrast"] = cluster_ci(\n                (contrast_accepted[y == 0].astype(float)-accepted[y == 0]), np.array(g)[y == 0], lambda a: float(a.mean()))\n            # Descriptive test risk-coverage points; threshold selection remains validation-only.\n            probe["risk_coverage"] = []\n            for threshold in np.linspace(0., 1., 21):\n                keep = np.array([not (r["margin"] > 0) or risk[r["id"]] <= threshold for r in rows])\n                errors = np.array([int((r["margin"] > 0) != bool(r["label"])) for r in rows])\n                probe["risk_coverage"].append({"threshold": float(threshold), "n_answered": int(keep.sum()),\n                    "coverage": cluster_ci(keep.astype(float), g, lambda a: float(a.mean())),\n                    "risk": cluster_ci(errors[keep].astype(float), np.array(g)[keep], lambda a: float(a.mean()))})\n            out["probe"] = probe\n        metrics["tasks"][task] = out\n    atomic_json(Path(root) / cfg.model / "metrics.json", metrics)\n    return metrics\n\n\ndef geometry_and_transfer(root, cfg, manifest, index, environment, unembedding):\n    """Secondary analysis only, frozen-layer transfer. No test choice feeds main analysis."""\n    import numpy as np\n    import joblib\n    freeze = read_json(Path(root) / cfg.model / "analysis_freeze.json")\n    path = Path(root) / cfg.model / "analysis_bundle.joblib"\n    if file_hash(path) != freeze["bundle_sha256"]:\n        raise RuntimeError("Bundle hash mismatch")\n    bundle = joblib.load(path)\n    cal = bundle["calibration"]\n    test = load_completed(root, cfg, manifest, index, environment, "test")\n    result = {"geometry_split": "calibration", "transfer_split": "test", "transfer": {}, "geometry": []}\n    for source, target in ((TASKS[0], TASKS[1]), (TASKS[1], TASKS[0])):\n        train_bundle = bundle["bundles"][source]\n        ceiling_bundle = bundle["bundles"][target]\n        rows = [r for r in test if r["task"] == target and r["margin"] > 0]\n        if not rows or train_bundle["status"] != "fitted" or ceiling_bundle["status"] != "fitted":\n            result["transfer"][source] = {"status": "underpowered_missing_class"}\n            continue\n        y, g = np.array([1-r["label"] for r in rows]), [r["source_group"] for r in rows]\n        scores = probe_scores(train_bundle, cal, rows)["probe_only"]\n        ceiling = probe_scores(ceiling_bundle, cal, rows)["probe_only"]\n        data = np.column_stack([y, scores, ceiling])\n        def ratio(a):\n            ta, ca = safe_auc(a[:, 0], a[:, 1]), safe_auc(a[:, 0], a[:, 2])\n            return (ta-.5)/(ca-.5) if ta is not None and ca is not None and ca > .5 else None\n        result["transfer"][source] = {"target": target, "auroc": auc_ci(y, scores, g),\n            "in_task_ceiling_auroc": auc_ci(y, ceiling, g), "above_chance_ratio": cluster_ci(data, g, ratio),\n            "n_false_yes": int(y.sum()), "power": "underpowered" if y.sum() < 40 else "at_least_40_false_yes"}\n    def cosine(a, b):\n        denominator = np.linalg.norm(a)*np.linalg.norm(b)\n        return float(a@b/denominator) if denominator > 0 else None\n    pool = [[r for r in cal if r["task"] == task and r["margin"] > 0] for task in TASKS[:2]]\n    if all(rows and len({r["label"] for r in rows}) == 2 for rows in pool):\n        for layer in range(pool[0][0]["hidden"].shape[0]):\n            xs = [np.stack([r["hidden"][layer] for r in rows]) for rows in pool]\n            ys = [np.array([1-r["label"] for r in rows]) for rows in pool]\n            directions = [x[y == 1].mean(0)-x[y == 0].mean(0) for x, y in zip(xs, ys)]\n            rng = np.random.default_rng(SEED + layer)\n            null = []\n            for _ in range(2000):\n                shuffled = [rng.permutation(y) for y in ys]\n                ds = [x[y == 1].mean(0)-x[y == 0].mean(0) for x, y in zip(xs, shuffled)]\n                value = cosine(*ds)\n                if value is not None:\n                    null.append(value)\n            # Bootstrap both task directions with the same source-component draw.\n            joined = [dict(r, task_index=i) for i, rows in enumerate(pool) for r in rows]\n            values = np.arange(len(joined))\n            def geometry_stat(indices, reference=None):\n                ds = []\n                for i in range(2):\n                    selected = [joined[int(j)] for j in indices if joined[int(j)]["task_index"] == i]\n                    if len({r["label"] for r in selected}) < 2:\n                        return None\n                    a = np.stack([r["hidden"][layer] for r in selected if r["label"] == 0]).mean(0)\n                    b = np.stack([r["hidden"][layer] for r in selected if r["label"] == 1]).mean(0)\n                    ds.append(a-b)\n                return cosine(*ds) if reference is None else cosine(ds[reference], np.asarray(unembedding))\n            groups = [r["source_group"] for r in joined]\n            result["geometry"].append({"layer": layer,\n                "cross_task_cosine": cluster_ci(values, groups, geometry_stat),\n                "null_interval95": np.quantile(null, [.025, .975]).tolist() if null else None,\n                "unembedding_cosines": [cluster_ci(values, groups, lambda a, i=i: geometry_stat(a, i)) for i in range(2)],\n                "n_yes": [len(r) for r in pool], "n_false_yes": [int(y.sum()) for y in ys],\n                "power": ["underpowered" if y.sum() < 40 else "at_least_40_false_yes" for y in ys]})\n    atomic_json(Path(root) / cfg.model / "secondary_metrics.json", result)\n    return result\n', 'avh_kaggle.py': '"""Private Kaggle transport and deterministic lean-study contracts. CPU import only.\n\nThe original Colab manifest/media are never overwritten. Prediction shards are\nprivate research artifacts. A test prediction is not permission to analyze it.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport collections\nimport concurrent.futures\nimport datetime\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport zipfile\n\nimport avh_common as avh\n\nPASSES = ("full", "cf0", "cf1", "blank", "no_media")\nSCHEMA = "counterprior-kaggle-v1"\n\n\ndef immutable_json(path, value):\n    path = Path(path)\n    if path.exists() and avh.read_json(path) != value:\n        raise RuntimeError(f"Refusing to change frozen record: {path}")\n    if not path.exists():\n        avh.atomic_json(path, value)\n    return value\n\n\ndef worker_for(question_id, n_workers):\n    if n_workers < 1:\n        raise ValueError("Positive worker count required")\n    return int(avh.digest(str(question_id))[:16], 16) % n_workers\n\n\ndef account_workers(account_index, n_workers):\n    if n_workers < 2 or n_workers % 2 or not 0 <= account_index < n_workers // 2:\n        raise ValueError("Require two GPUs per account and a valid account index")\n    return (account_index * 2, account_index * 2 + 1)\n\n\ndef passes_for(row):\n    return ("full", "no_media") if row["task"] == avh.TASKS[2] else PASSES\n\n\ndef stratified_take(rows, n, seed):\n    """Largest-remainder task x label allocation; fixed hash order within strata."""\n    if n > len(rows) or n < 1:\n        raise ValueError("Sample size exceeds the available source-disjoint pool")\n    strata = collections.defaultdict(list)\n    for row in rows:\n        strata[(row["task"], row["label"])].append(row)\n    target = {s: n * len(items) / len(rows) for s, items in strata.items()}\n    counts = {s: int(value) for s, value in target.items()}\n    order = sorted(strata, key=lambda s: (-(target[s] - counts[s]), s))\n    for s in order[:n - sum(counts.values())]:\n        counts[s] += 1\n    chosen = []\n    for s in sorted(strata):\n        chosen.extend(sorted(strata[s], key=lambda r: avh.digest([seed, "sample", r["id"]]))[:counts[s]])\n    return sorted(chosen, key=lambda r: r["id"])\n\n\ndef lean_manifest(parent, n=2000, model="gemma", seed=avh.SEED):\n    """New manifest; old split is retained as provenance, not silently overwritten.\n\n    All released groups receive one fixed 20/20/60 assignment independent of n.\n    Sampling happens inside each group-disjoint pool. Same-hash donor rule is\n    recomputed within these new pools, using the first two of four seeded donors.\n    """\n    import numpy as np\n    if model not in avh.MODEL_IDS or n not in (1000, 2000, 3000):\n        raise ValueError("Only the requested Gemma/Qwen lean sizes are supported")\n    if (model == "gemma" and n not in (2000, 3000)) or (model == "qwen" and n != 1000):\n        raise ValueError("Gemma n=2000/3000; optional Qwen n=1000")\n    rows = parent["rows"]\n    if len({r["id"] for r in rows}) != len(rows):\n        raise ValueError("Parent manifest contains duplicate question IDs")\n    groups = sorted({r["source_group"] for r in rows})\n    order = np.random.default_rng(seed).permutation(groups).tolist()\n    a, b = int(.2 * len(order)), int(.4 * len(order))\n    assignment = {g: "calibration" if i < a else "validation" if i < b else "test"\n                  for i, g in enumerate(order)}\n    sizes = {"calibration": n // 5, "validation": n // 5, "test": n * 3 // 5}\n    selected = []\n    for split, count in sizes.items():\n        pool = [r for r in rows if assignment[r["source_group"]] == split]\n        selected.extend(dict(r, parent_split=r["split"], split=split)\n                        for r in stratified_take(pool, count, seed))\n    video_meta = {r["video_id"]: r for r in rows}\n    for row in selected:\n        candidates = sorted(v for v, meta in video_meta.items()\n                            if assignment[meta["source_group"]] == row["split"]\n                            and meta["source_group"] != row["source_group"] and not meta["is_swapped"])\n        if len(candidates) < 4:\n            raise ValueError("Need four candidate donor clips in each new split")\n        rng = np.random.default_rng(int(avh.digest([seed, row["video_id"], row["task"]])[:16], 16))\n        row["donors"] = rng.choice(candidates, 4, replace=False).tolist()[:2]\n    return {"schema": SCHEMA, "model": model, "n": n, "seed": seed, "k": 2,\n        "parent_manifest_sha256": avh.digest(parent), "group_assignment_sha256": avh.digest(assignment),\n        "split_counts": sizes, "test_scoring_policy": "sealed_precompute",\n        "limitations": parent.get("limitations", []), "object_rule": parent.get("object_rule"),\n        "sampling_rule": "group-disjoint 20/20/60 pools; proportional task x label largest remainder; seeded hash order",\n        "donor_rule": "same-split released groups, seeded first two of four; unverified swap status explicitly retained",\n        "rows": sorted(selected, key=lambda r: r["id"])}\n\n\ndef portable_index(index, media_root):\n    """Never change media bytes or the source index; create a relocatable companion."""\n    result = {}\n    for vid, entry in index.items():\n        directory = Path(entry["directory"])\n        target = Path(media_root) / "media" / directory.parent.name / directory.name\n        if not target.is_dir():\n            raise FileNotFoundError(target)\n        result[vid] = dict(entry, directory=target.relative_to(media_root).as_posix())\n    return result\n\n\ndef resolved_index(dataset_root):\n    root = Path(dataset_root).resolve()\n    index = avh.read_json(root / "kaggle_media_index.json")\n    result = {}\n    for vid, entry in index.items():\n        directory = (root / entry["directory"]).resolve()\n        if not directory.is_relative_to(root):\n            raise ValueError("Media path escapes the private dataset")\n        result[vid] = dict(entry, directory=str(directory))\n    return result\n\n\ndef export_private_study(root, output, support_dir, code_git_hash, code_dirty=True, read_workers=8, local_staging=None):\n    """Run on CPU Colab. Zip prepared bytes, evidence, old results and new support.\n\n    Does not delete/edit Drive files or upload anything. No tokens or HF cache.\n    Bounded concurrent reads hide Drive\'s per-file latency. The ZIP is built on\n    local CPU disk, then copied to Drive in one sequential transfer. A cleanly\n    interrupted older export can contribute verified entries without re-reading\n    them individually from Drive. Never use a GPU/TPU for this operation.\n    """\n    root, output, support = Path(root), Path(output), Path(support_dir)\n    if output.exists():\n        raise FileExistsError("Preserve the existing export; choose a new version filename")\n    if len(code_git_hash) != 40 or any(c not in "0123456789abcdef" for c in code_git_hash):\n        raise ValueError("Real source repository commit hash required")\n    index = avh.read_json(root / "media_index_f8.json")\n    pindex = portable_index(index, root)\n    files, media_hashes = {}, {}\n    required = ["manifest.json", "media_index_f8.json", "lineage.json", "terms_review.json",\n        "release_inventory.json", "released_qa.json", "qa_resolution.json", "audit.csv"]\n    for model in avh.MODEL_IDS:\n        required += [f"{model}/model_metadata.json", f"{model}/scoring_amendment.json"]\n    terms = avh.read_json(root / "terms_review.json")\n    required.extend(entry["evidence_file"] for entry in terms.values())\n    for name in sorted(set(required)):\n        source = (root / name).resolve()\n        if not source.is_relative_to(root.resolve()) or not source.is_file():\n            raise ValueError(f"Missing/unsafe study evidence: {name}")\n        files[name] = source\n    for model in avh.MODEL_IDS:\n        amendment = avh.read_json(root / model / "scoring_amendment.json")\n        name = f"{model}/{amendment[\'prefix_evidence_file\']}"\n        files[name] = root / name\n        for name in ("pre_smoke.json", "smoke_f8.json", "environment.json", "yes_no_unembedding.npy"):\n            path = root / model / name\n            if path.is_file():\n                files[f"prior_colab/{model}/{name}"] = path\n    for vid, entry in pindex.items():\n        for name, expected in entry["file_hashes"].items():\n            source = root / entry["directory"] / name\n            archive_name = f"{entry[\'directory\']}/{name}"\n            files[archive_name] = source\n            media_hashes[archive_name] = expected\n    prereg = support / "PREREGISTRATION_KAGGLE.md"\n    if not prereg.is_file():\n        raise FileNotFoundError(prereg)\n    files["runtime/PREREGISTRATION_KAGGLE.md"] = prereg\n    for name in ("PREREGISTRATION.md", "lineage_rules.json", "publication_policy.json", "shared_data_summary.json"):\n        if (root / name).is_file():\n            files[f"prior_colab/{name}"] = root / name\n    manifest = avh.read_json(root / "manifest.json")\n    generated = {"kaggle_media_index.json": pindex,\n        "lean_manifest_n2000.json": lean_manifest(manifest, n=2000),\n        "lean_manifest_n3000.json": lean_manifest(manifest, n=3000)}\n    generated_bytes = {name: json.dumps(value, indent=2).encode("utf-8") for name, value in generated.items()}\n    hashes = {name: media_hashes.get(name) or avh.file_hash(path) for name, path in files.items()}\n    hashes.update({name: hashlib.sha256(value).hexdigest() for name, value in generated_bytes.items()})\n    payload = {"schema": SCHEMA, "private_only": True, "code_git_hash": code_git_hash,\n        "code_dirty": bool(code_dirty), "parent_manifest_sha256": avh.digest(manifest),\n        "original_media_index_sha256": avh.digest(index),\n        "files": hashes}\n    if not 1 <= read_workers <= 16:\n        raise ValueError("CPU read concurrency must be between 1 and 16")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    local = Path(local_staging) if local_staging else (\n        Path("/content/counterprior-export-cache") if Path("/content").is_dir() else output.parent / ".cpu-export-staging")\n    local.mkdir(parents=True, exist_ok=True)\n    temporary = local / (output.name + ".building")\n    # Keep previous local/Drive partials intact; only final verified bytes are reused.\n    candidates = [output.with_suffix(output.suffix + ".partial"), temporary]\n    reusable = []\n    for i, source in enumerate(candidates):\n        if source.is_file() and zipfile.is_zipfile(source):\n            copied = local / f"recovery-{i}-{output.name}"\n            shutil.copyfile(source, copied)\n            reusable.append(copied)\n    complete, reused = set(), 0\n    tick = time.monotonic()\n    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as z:\n        for source in reusable:\n            with zipfile.ZipFile(source) as previous:\n                for info in previous.infolist():\n                    if info.filename not in files or info.filename in complete:\n                        continue\n                    try:\n                        data = previous.read(info)\n                    except (zipfile.BadZipFile, RuntimeError, EOFError):\n                        continue\n                    if hashlib.sha256(data).hexdigest() != hashes[info.filename]:\n                        continue\n                    z.writestr(info.filename, data)\n                    complete.add(info.filename)\n                    reused += 1\n        print(f"Reused {reused} verified files; CPU read workers={read_workers}. Building ZIP locally.", flush=True)\n\n        def read_verified(item):\n            name, path = item\n            data = path.read_bytes()\n            if hashlib.sha256(data).hexdigest() != hashes[name]:\n                raise RuntimeError(f"Prepared bytes changed while exporting: {name}")\n            return name, data\n\n        remaining = iter((name, path) for name, path in sorted(files.items()) if name not in complete)\n        pending = collections.deque()\n        with concurrent.futures.ThreadPoolExecutor(max_workers=read_workers) as pool:\n            for _ in range(read_workers * 2):\n                item = next(remaining, None)\n                if item is not None:\n                    pending.append(pool.submit(read_verified, item))\n            while pending:\n                name, data = pending.popleft().result()\n                z.writestr(name, data)\n                complete.add(name)\n                if len(complete) % 500 == 0:\n                    print(f"Packed {len(complete)}/{len(files)} files; elapsed={time.monotonic()-tick:.1f}s", flush=True)\n                item = next(remaining, None)\n                if item is not None:\n                    pending.append(pool.submit(read_verified, item))\n        for name, data in generated_bytes.items():\n            z.writestr(name, data)\n        z.writestr("kaggle_bundle.json", json.dumps(payload, indent=2))\n        z.writestr("cpu_verified.json", json.dumps({"schema": SCHEMA, "status": "cpu_verified",\n            "bundle_sha256": avh.digest(payload), "media_index_sha256": avh.digest(pindex),\n            "verification": "Prepared bytes and evidence checked on CPU during private archive export"}, indent=2))\n        z.writestr("run_control.json", json.dumps({"mode": "auto", "model": "gemma", "n": 2000,\n            "n_workers": 6, "reference": "fresh_unbatched_gpu", "sample_policy": "calibration_addon",\n            "gemma_complete": False, "qwen_attempt": 0}, indent=2))\n        z.writestr("CHECKSUMS.sha256", "".join(f"{h}  {name}\\n" for name, h in sorted(hashes.items())))\n    local_sha = avh.file_hash(temporary)\n    print("Local ZIP complete. Copying one archive to Drive...", flush=True)\n    transfer = output.with_suffix(output.suffix + ".transferring")\n    shutil.copyfile(temporary, transfer)\n    transfer.replace(output)\n    result = {"archive": str(output), "bytes": output.stat().st_size,\n              "sha256": local_sha, "media_clips": len(index), "private_only": True,\n              "cpu_read_workers": read_workers, "reused_files": reused,\n              "export_method": "bounded parallel CPU reads, local ZIP, one Drive copy"}\n    avh.atomic_json(output.with_suffix(".checksum.json"), result)\n    return result\n\n\ndef verify_bundle(root):\n    """CPU-only integrity pass before allocating GPUs. No Drive traversal at runtime."""\n    root = Path(root)\n    bundle = avh.read_json(root / "kaggle_bundle.json")\n    if bundle.get("schema") != SCHEMA or bundle.get("private_only") is not True:\n        raise ValueError("Wrong study bundle or public-media policy")\n    for i, (name, expected) in enumerate(sorted(bundle["files"].items()), 1):\n        path = (root / name).resolve()\n        if not path.is_relative_to(root.resolve()) or avh.file_hash(path) != expected:\n            raise RuntimeError(f"Bundle integrity failure: {name}")\n        if i % 500 == 0:\n            print(f"Verified {i}/{len(bundle[\'files\'])} private files", flush=True)\n    index = resolved_index(root)\n    avh.verify_media(index)\n    return {"schema": SCHEMA, "bundle_sha256": avh.digest(bundle),\n            "media_index_sha256": avh.digest(avh.read_json(root / "kaggle_media_index.json")),\n            "status": "cpu_verified", "private_only": True}\n\n\ndef safe_pairs(manifest, splits=None):\n    return {(r["id"], p) for r in manifest["rows"] if splits is None or r["split"] in splits\n            for p in passes_for(r)}\n\n\nclass PairJournal:\n    """Per-worker pair checkpoints; full-only fp16 NPY shards of 250 questions.\n\n    Buffers are fsynced every 25 completed questions and on normal/soft-stop exit.\n    An abrupt VM loss can repeat up to 25 uncommitted questions. A saved Kaggle\n    output version must be reattached on restart; /working is not a shared drive.\n    """\n    def __init__(self, directory, identity, worker_id, account_index, n_workers, manifest):\n        self.directory = Path(directory)\n        self.identity, self.worker_id, self.account_index = identity, worker_id, account_index\n        self.n_workers, self.manifest = n_workers, manifest\n        if worker_id not in account_workers(account_index, n_workers):\n            raise ValueError("Worker ID is not assigned to this account")\n\n    def __enter__(self):\n        self.directory.mkdir(parents=True, exist_ok=True)\n        self.lock = self.directory / "worker.lock"\n        fd = os.open(self.lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)\n        os.close(fd)\n        try:\n            immutable_json(self.directory / "identity.json", self.identity)\n            self.path = self.directory / "predictions.jsonl"\n            self.rows, valid = [], 0\n            lines = self.path.read_bytes().splitlines(keepends=True) if self.path.exists() else []\n            for i, line in enumerate(lines):\n                try:\n                    row = json.loads(line)\n                except (ValueError, UnicodeDecodeError):\n                    if i != len(lines) - 1 or line.endswith(b"\\n"):\n                        raise RuntimeError("Corrupt journal interior")\n                    break\n                if row["identity_sha256"] != avh.digest(self.identity):\n                    raise RuntimeError("Checkpoint identity changed")\n                self.rows.append(row)\n                valid += len(line)\n            self.done = {(r["question_id"], r["pass"]) for r in self.rows}\n            if len(self.done) != len(self.rows):\n                raise RuntimeError("Duplicate question/pass pair in checkpoint")\n            wanted = safe_pairs(self.manifest)\n            if not self.done <= wanted or any(worker_for(q, self.n_workers) != self.worker_id for q, _ in self.done):\n                raise RuntimeError("Checkpoint contains unassigned work")\n            if self.path.exists():\n                with self.path.open("r+b") as f:\n                    f.truncate(valid)\n                if valid and not self.path.read_bytes().endswith(b"\\n"):\n                    with self.path.open("ab") as f:\n                        f.write(b"\\n")\n            self.full_count = sum(r["pass"] == "full" for r in self.rows)\n            self.maps, self.dirty_questions = {}, 0\n            self.handle = self.path.open("a", encoding="utf-8")\n            return self\n        except BaseException:\n            self.lock.unlink(missing_ok=True)\n            raise\n\n    def append(self, row, pass_name, result, hidden=None):\n        import numpy as np\n        key = (row["id"], pass_name)\n        if key in self.done:\n            return\n        if pass_name not in passes_for(row) or worker_for(row["id"], self.n_workers) != self.worker_id:\n            raise ValueError("Unassigned question/pass")\n        record = dict(result, question_id=row["id"], id=row["id"], split=row["split"],\n            **{"pass": pass_name}, worker_id=self.worker_id, account_index=self.account_index,\n            model=self.identity["model"], model_revision=self.identity["model_revision"],\n            code_git_hash=self.identity["code_git_hash"], code_sha256=self.identity["code_sha256"],\n            code_hash=self.identity["code_sha256"],\n            timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),\n            identity_sha256=avh.digest(self.identity))\n        if pass_name == "full":\n            h = np.asarray(hidden, dtype=np.float16)\n            if h.ndim != 2 or not np.isfinite(h).all():\n                raise ValueError("Full pass requires finite all-layer hidden states")\n            number, slot = divmod(self.full_count, 250)\n            name = f"full-hidden-{number:05d}.npy"\n            if name not in self.maps:\n                path = self.directory / name\n                self.maps[name] = np.lib.format.open_memmap(path, mode="r+" if path.exists() else "w+",\n                    dtype=np.float16, shape=(250, *h.shape))\n            self.maps[name][slot] = h\n            record.update(feature=name, feature_slot=slot,\n                feature_value_sha256=hashlib.sha256(h.tobytes()).hexdigest())\n            self.full_count += 1\n        elif hidden is not None:\n            raise ValueError("Hidden states may be saved only for the full pass")\n        self.handle.write(json.dumps(record, allow_nan=False) + "\\n")\n        self.rows.append(record)\n        self.done.add(key)\n\n    def question_complete(self):\n        self.dirty_questions += 1\n        if self.dirty_questions >= 25:\n            self.flush()\n\n    def flush(self):\n        for array in self.maps.values():\n            array.flush()\n        self.handle.flush()\n        os.fsync(self.handle.fileno())\n        self.dirty_questions = 0\n\n    def __exit__(self, *args):\n        try:\n            self.flush()\n            self.handle.close()\n        finally:\n            self.maps.clear()\n            self.lock.unlink(missing_ok=True)\n\n\ndef validate_outputs(directories, manifest, identity, n_workers, splits=None):\n    """Coverage-only merge gate. Never computes test outcomes or chooses settings."""\n    import numpy as np\n    wanted = safe_pairs(manifest, splits)\n    found, records, locations = set(), [], set()\n    lookup = {r["id"]: r for r in manifest["rows"]}\n    hidden_shape = None\n    for directory in map(Path, directories):\n        if avh.read_json(directory / "identity.json") != identity:\n            raise RuntimeError("Mixed runtime/study identity")\n        for line in (directory / "predictions.jsonl").read_text().splitlines():\n            r = json.loads(line)\n            pair = (r["question_id"], r["pass"])\n            if pair not in wanted or pair in found:\n                raise RuntimeError("Duplicate or unexpected question/pass pair")\n            if (r["identity_sha256"] != avh.digest(identity)\n                    or worker_for(pair[0], n_workers) != r["worker_id"]\n                    or r["worker_id"] not in account_workers(r["account_index"], n_workers)):\n                raise RuntimeError("Worker assignment or checkpoint identity mismatch")\n            expected = {"model": identity["model"], "model_revision": identity["model_revision"],\n                        "code_git_hash": identity["code_git_hash"], "code_sha256": identity["code_sha256"],\n                        "code_hash": identity["code_sha256"], "split": lookup[pair[0]]["split"]}\n            if any(r.get(k) != value for k, value in expected.items()):\n                raise RuntimeError("Per-row provenance/split differs from the frozen assignment")\n            if not np.isfinite([r["margin"], r["yes_logsumexp"], r["no_logsumexp"]]).all():\n                raise FloatingPointError("Nonfinite merged answer scores")\n            if r["answer"] != int(r["margin"] > 0) or abs(r["margin"] - (r["yes_logsumexp"] - r["no_logsumexp"])) > 1e-5:\n                raise RuntimeError("Saved answer/margin/logit disagreement")\n            if datetime.datetime.fromisoformat(r["timestamp"]).tzinfo is None:\n                raise RuntimeError("Timestamp requires a timezone")\n            if r["pass"] == "full":\n                feature = (directory / r["feature"]).resolve()\n                if not feature.is_relative_to(directory.resolve()):\n                    raise ValueError("Unsafe feature path")\n                location = (str(feature), r["feature_slot"])\n                if not isinstance(r["feature_slot"], int) or not 0 <= r["feature_slot"] < 250 or location in locations:\n                    raise RuntimeError("Invalid or aliased feature slot")\n                locations.add(location)\n                h = np.load(feature, mmap_mode="r", allow_pickle=False)[r["feature_slot"]]\n                if h.dtype != np.float16 or not np.isfinite(h).all() or hashlib.sha256(h.tobytes()).hexdigest() != r["feature_value_sha256"]:\n                    raise RuntimeError("Feature data changed")\n                if h.ndim != 2 or (hidden_shape is not None and h.shape != hidden_shape):\n                    raise RuntimeError("Inconsistent all-layer hidden-state shape")\n                hidden_shape = h.shape\n            found.add(pair)\n            records.append((directory, r))\n    if found != wanted:\n        raise RuntimeError(f"Incomplete coverage: {len(wanted - found)} missing pairs")\n    return records\n\n\ndef timing_summary(reports, manifest, n_workers):\n    """Observed timings + explicitly labelled planning forecasts, never metrics.json.\n\n    Per-pass allocation divides a measured batch duration by its pass count; it is\n    not an isolated-pass benchmark. Forecast assumes equal workers/no failures;\n    compute-hours and end-to-end wall time remain distinct.\n    """\n    import numpy as np\n    rows = [r for report in reports for r in report["rows"]]\n    if len({r["question_id"] for r in rows}) != 50:\n        raise RuntimeError("Timing check must cover 50 distinct calibration questions")\n    identities = {avh.digest(r["identity"]) for r in reports}\n    if len(identities) != 1:\n        raise RuntimeError("Timing workers use different numerical environments")\n    observed = {}\n    for task in avh.TASKS:\n        selected = [r for r in rows if r["task"] == task]\n        if not selected:\n            raise RuntimeError("Timing check missing a task stratum")\n        values = [r["question_seconds"] for r in selected]\n        groups = [r["source_group"] for r in selected]\n        observed[task] = {"question_seconds_mean": avh.cluster_ci(values, groups, lambda x: float(x.mean())),\n            "question_seconds_p50": avh.cluster_ci(values, groups, lambda x: float(np.quantile(x, .5))),\n            "question_seconds_p90": avh.cluster_ci(values, groups, lambda x: float(np.quantile(x, .9)))}\n    counts = collections.Counter(r["task"] for r in manifest["rows"])\n    compute_seconds = sum(counts[t] * observed[t]["question_seconds_mean"]["estimate"] for t in counts)\n    batch_seconds = [b["seconds"] for r in rows for b in r["batches"]]\n    allocations = collections.defaultdict(list)\n    for r in rows:\n        for b in r["batches"]:\n            for p in b["passes"]:\n                allocations[p].append(b["seconds"] / len(b["passes"]))\n    return {"kind": "planning_forecast_not_research_result", "n_questions": 50,\n        "n_source_groups": len({r["source_group"] for r in rows}), "observed": observed,\n        "batch_seconds": {"p50": float(np.quantile(batch_seconds, .5)), "p90": float(np.quantile(batch_seconds, .9)),\n                          "n_batches": len(batch_seconds)},\n        "allocated_pass_seconds": {p: {"p50": float(np.quantile(v, .5)), "p90": float(np.quantile(v, .9)),\n                                              "n": len(v)} for p, v in allocations.items()},\n        "batch_sizes": [r["batch_size"] for r in reports],\n        "model_load_seconds_observed": [r["load_seconds"] for r in reports],\n        "forecast": {"gpu_hours_scoring_only": compute_seconds / 3600,\n                     "ideal_wall_hours_scoring_only": compute_seconds / (3600 * n_workers),\n                     "n_workers": n_workers, "question_count": len(manifest["rows"]),\n                     "assumptions": "representative task timing, all workers equally fast; excludes startup, retries, transfers, audit, analysis and site"},\n        "warnings": ["Per-pass timings are allocations of batch wall time, not isolated-pass measurements",\n                     "Task-specific timing CIs are empirical; the wall-time forecast has no validated end-to-end CI",\n                     "No predicted accuracy, AUROC, false-yes rate or site result"]}\n\n\ndef restore_checkpoint(source, destination, worker_id, bundle_sha256):\n    """Attach the participant\'s own previous private output version before resuming."""\n    source, destination = Path(source), Path(destination)\n    identity = avh.read_json(source / "identity.json")\n    if identity["bundle_sha256"] != bundle_sha256:\n        raise RuntimeError("Resume checkpoint belongs to another private data version")\n    for line in (source / "predictions.jsonl").read_text().splitlines():\n        if json.loads(line)["worker_id"] != worker_id:\n            raise RuntimeError("Resume checkpoint belongs to another worker")\n    if destination.exists():\n        raise FileExistsError("Do not overwrite a live or existing checkpoint")\n    destination.mkdir(parents=True)\n    for path in source.iterdir():\n        if path.is_file() and path.name != "worker.lock":\n            shutil.copyfile(path, destination / path.name)\n\n\ndef calibration_power_decision(manifest, records, expanded, policy):\n    """Predeclared one-time planning decision; never uses validation/test predictions."""\n    if policy == "fixed_2000":\n        return {"policy": policy, "chosen_n": 2000, "adaptation": "disabled by owner decision"}\n    if policy not in ("adaptive_calibration_first", "calibration_addon"):\n        raise RuntimeError("Sample policy pending owner decision")\n    if manifest.get("n") != 2000:\n        raise RuntimeError("The add-on decision is permitted once, from the initial n=2000 calibration")\n    lookup = {r["id"]: r for r in manifest["rows"]}\n    full = {}\n    for _, result in records:\n        if result["pass"] != "full":\n            continue\n        row = lookup[result["question_id"]]\n        if row["split"] != "calibration":\n            raise RuntimeError("Sample expansion decision cannot inspect validation/test predictions")\n        if row["id"] in full:\n            raise RuntimeError("Duplicate calibration question")\n        full[row["id"]] = result\n    required = {r["id"] for r in manifest["rows"] if r["split"] == "calibration"}\n    if set(full) != required:\n        raise RuntimeError("Complete calibration before one-time expansion decision")\n    planning = {}\n    for task in avh.TASKS[:2]:\n        negatives = [r for r in manifest["rows"] if r["split"] == "calibration"\n                     and r["task"] == task and r["label"] == 0]\n        if not negatives:\n            raise RuntimeError("Missing negative calibration stratum")\n        rate = avh.cluster_ci([full[r["id"]]["answer"] for r in negatives],\n            [r["source_group"] for r in negatives], lambda a: float(a.mean()))\n        test_negatives = sum(r["split"] == "test" and r["task"] == task and r["label"] == 0\n                             for r in manifest["rows"])\n        planning[task] = {"calibration_false_yes_rate": rate,\n            "test_negative_inventory": test_negatives,\n            "expected_false_yes_planning_only": rate["estimate"] * test_negatives}\n    enlarge = any(r["expected_false_yes_planning_only"] < 40 for r in planning.values())\n    if enlarge and (expanded["n"] != 3000 or expanded["group_assignment_sha256"] != manifest["group_assignment_sha256"]):\n        raise RuntimeError("Expanded sample must preserve the group assignment and n<=3000")\n    return {"kind": "calibration_based_sample_planning_not_test_results", "policy": policy,\n        "chosen_n": 3000 if enlarge else 2000, "max_adaptations": 1, "planning": planning,\n        "rule": "Increase once if either primary task expects fewer than 40 test false-yes cases; never inspect test predictions"}\n\n\ndef calibration_receipt(journal, manifest, worker_id, n_workers, launch_id):\n    """Small calibration-only receipt, safe to copy from a running private notebook."""\n    import numpy as np\n    items = sorted((r for r in manifest["rows"] if r["split"] == "calibration"\n                    and worker_for(r["id"], n_workers) == worker_id), key=lambda r: r["id"])\n    required = {(r["id"], p) for r in items for p in passes_for(r)}\n    if not required <= journal.done:\n        return None\n    journal.flush()\n    lookup = {r["question_id"]: r["answer"] for r in journal.rows if r["pass"] == "full"}\n    packed = np.packbits(np.asarray([lookup[r["id"]] for r in items], dtype=np.uint8)).tobytes().hex()\n    return {"worker_id": worker_id, "n_workers": n_workers, "launch_id": launch_id,\n        "manifest_sha256": avh.digest(manifest), "identity_sha256": avh.digest(journal.identity),\n        "cal_ids_sha256": avh.digest([r["id"] for r in items]), "n_cal": len(items),\n        "cal_pair_count": len(required), "answers_packed_hex": packed, "scope": "calibration_only"}\n\n\ndef records_from_calibration_receipts(accounts, manifest, n_workers):\n    import numpy as np\n    receipts = [r for account in accounts for r in account]\n    if len(receipts) != n_workers or {r["worker_id"] for r in receipts} != set(range(n_workers)):\n        raise RuntimeError("Need exactly one calibration receipt per worker")\n    if len({r["identity_sha256"] for r in receipts}) != 1:\n        raise RuntimeError("Calibration receipts mix code/data/runtime identities")\n    records = []\n    for receipt in receipts:\n        rows = sorted((r for r in manifest["rows"] if r["split"] == "calibration"\n                       and worker_for(r["id"], n_workers) == receipt["worker_id"]), key=lambda r: r["id"])\n        if (receipt["scope"] != "calibration_only" or receipt["manifest_sha256"] != avh.digest(manifest)\n                or receipt["cal_ids_sha256"] != avh.digest([r["id"] for r in rows])\n                or receipt["n_cal"] != len(rows)\n                or receipt["cal_pair_count"] != sum(len(passes_for(r)) for r in rows)):\n            raise RuntimeError("Calibration receipt coverage does not match the frozen sample")\n        bits = np.unpackbits(np.frombuffer(bytes.fromhex(receipt["answers_packed_hex"]), dtype=np.uint8))\n        if len(bits) < len(rows):\n            raise RuntimeError("Truncated calibration answer receipt")\n        records.extend((None, {"question_id": r["id"], "pass": "full", "answer": int(a)})\n                       for r, a in zip(rows, bits))\n    return records\n\n\ndef calibration_records_from_outputs(directories, manifest, n_workers):\n    """Inspect calibration pairs only; ignore sealed validation/test values."""\n    wanted = safe_pairs(manifest, ("calibration",))\n    seen, records, identities = set(), [], set()\n    family = {k: manifest[k] for k in ("parent_manifest_sha256", "group_assignment_sha256", "seed", "k")}\n    for directory in map(Path, directories):\n        identity = avh.read_json(directory / "identity.json")\n        if identity["selection_family"] != family or identity["model"] != "gemma" or identity["n_workers"] != n_workers:\n            raise RuntimeError("Calibration outputs belong to another study")\n        identities.add(avh.digest(identity))\n        for line in (directory / "predictions.jsonl").read_text().splitlines():\n            row = json.loads(line)\n            pair = (row["question_id"], row["pass"])\n            if pair not in wanted:\n                continue\n            if pair in seen or row["identity_sha256"] != avh.digest(identity) or worker_for(pair[0], n_workers) != row["worker_id"]:\n                raise RuntimeError("Duplicate or misassigned calibration pair")\n            seen.add(pair)\n            if pair[1] == "full":\n                records.append((directory, row))\n    if len(identities) != 1 or seen != wanted:\n        raise RuntimeError("Complete matching calibration outputs are required")\n    return records\n\n\ndef select_output_versions(directories):\n    """Select a cumulative resume version; retain disjoint base/add-on journals.\n\n    Overlap must be byte-equivalent JSON records from a genuine checkpoint resume.\n    A fresh rerun with new values/timestamps is not silently deduplicated.\n    """\n    selected = []\n    for source in map(Path, directories):\n        current = {(r["question_id"], r["pass"]): r for r in\n                   (json.loads(line) for line in (source / "predictions.jsonl").read_text().splitlines())}\n        keep = True\n        for old in list(selected):\n            previous = {(r["question_id"], r["pass"]): r for r in\n                        (json.loads(line) for line in (old / "predictions.jsonl").read_text().splitlines())}\n            overlap = current.keys() & previous.keys()\n            if not overlap:\n                continue\n            if any(current[k] != previous[k] for k in overlap):\n                raise RuntimeError("Overlapping runs differ; cannot silently deduplicate")\n            if current.keys() <= previous.keys():\n                keep = False\n                break\n            if previous.keys() <= current.keys():\n                selected.remove(old)\n            else:\n                raise RuntimeError("Partially overlapping output versions")\n        if keep:\n            selected.append(source)\n    return selected\n\n\ndef import_for_analysis(dataset, manifest, directories, destination, n_workers):\n    """Strict complete merge into the existing CPU analysis schema; sealed test gate stays active."""\n    dataset, destination = Path(dataset), Path(destination)\n    directories = list(map(Path, directories))\n    if not directories:\n        raise ValueError("No worker output directories supplied")\n    identity = avh.read_json(directories[0] / "identity.json")\n    family = {k: manifest[k] for k in ("parent_manifest_sha256", "group_assignment_sha256", "seed", "k")}\n    if identity["selection_family"] != family or identity["n_workers"] != n_workers:\n        raise RuntimeError("Wrong sample family or worker count")\n    actual_code = avh.digest({name: avh.file_hash(Path(__file__).parent / name)\n                            for name in ("avh_common.py", "avh_kaggle.py", "avh_kaggle_worker.py")})\n    if identity["code_sha256"] != actual_code:\n        raise RuntimeError("Analysis runtime differs from the scored code snapshot")\n    records = validate_outputs(directories, manifest, identity, n_workers)\n    model = identity["model"]\n    cfg = avh.Config(str(destination), model, identity["model_revision"], k=2,\n                     workers=n_workers, scoring_mode="full_forward")\n    for name in ("terms_review.json", "release_inventory.json", "gemma_LICENSE.txt", "qwen_LICENSE.txt"):\n        destination.mkdir(parents=True, exist_ok=True)\n        target = destination / name\n        if target.exists() and avh.file_hash(target) != avh.file_hash(dataset / name):\n            raise RuntimeError("Cannot replace existing analysis evidence")\n        shutil.copyfile(dataset / name, target)\n    out = destination / model\n    out.mkdir(exist_ok=True)\n    for name in ("model_metadata.json", "scoring_amendment.json"):\n        shutil.copyfile(dataset / model / name, out / name)\n    amendment = avh.read_json(out / "scoring_amendment.json")\n    shutil.copyfile(dataset / model / amendment["prefix_evidence_file"], out / amendment["prefix_evidence_file"])\n    prereg = dataset / "runtime/PREREGISTRATION_KAGGLE.md"\n    prereg_bytes = prereg.read_bytes()\n    if "serial_amendment_sha256" in identity:\n        amendment_path = Path(__file__).with_name("GEMMA_SERIAL_AMENDMENT.md")\n        if avh.file_hash(amendment_path) != identity["serial_amendment_sha256"]:\n            raise RuntimeError("Analysis amendment differs from the scored protocol")\n        prereg_bytes += b"\\n\\n" + amendment_path.read_bytes()\n    destination_prereg = destination / "PREREGISTRATION.md"\n    if destination_prereg.exists() and destination_prereg.read_bytes() != prereg_bytes:\n        raise RuntimeError("Do not overwrite existing analysis preregistration")\n    destination_prereg.write_bytes(prereg_bytes)\n    immutable_json(destination / "manifest.json", manifest)\n    index = resolved_index(dataset)\n    environment = identity["environment"]\n    signature = avh.study_signature(cfg, manifest, index, destination, environment)\n    lookup = {r["id"]: r for r in manifest["rows"]}\n    staged = collections.defaultdict(list)\n    copied, feature_hashes = {}, {}\n    for source, record in records:\n        row = lookup[record["question_id"]]\n        result = dict(record, signature=signature, **{k: row[k] for k in\n            ("id", "video_id", "source_group", "split", "task", "label", "object")})\n        if record["pass"] == "full":\n            key = (str(source), record["feature"])\n            if key not in feature_hashes:\n                feature_hashes[key] = avh.file_hash(source / record["feature"])\n            target = out / "f8/full/shards" / f"worker-{record[\'worker_id\']:02d}-{feature_hashes[key][:16]}-{record[\'feature\']}"\n            target.parent.mkdir(parents=True, exist_ok=True)\n            if key not in copied:\n                if target.exists() and avh.file_hash(target) != avh.file_hash(source / record["feature"]):\n                    raise RuntimeError("Existing merged feature shard changed")\n                shutil.copyfile(source / record["feature"], target)\n                copied[key] = target\n                if avh.file_hash(target) != feature_hashes[key]:\n                    raise RuntimeError("Merged feature copy changed")\n            result.update(feature=str(target.relative_to(out / "f8/full")),\n                          feature_index=record["feature_slot"], feature_sha256=feature_hashes[key])\n        staged[(record["pass"], record["worker_id"])].append(result)\n    for (name, worker), rows in staged.items():\n        target = out / "f8" / name / f"partition-{worker}-of-{n_workers}.jsonl"\n        data = "".join(json.dumps(r, allow_nan=False) + "\\n" for r in sorted(rows, key=lambda r: r["id"]))\n        if target.exists() and target.read_text() != data:\n            raise RuntimeError("Cannot replace an existing merged journal")\n        target.parent.mkdir(parents=True, exist_ok=True)\n        target.write_text(data)\n    avh.create_audit(destination, manifest)\n    immutable_json(out / "environment.json", environment)\n    immutable_json(out / "merge_receipt.json", {"status": "complete_coverage_only", "signature": signature,\n        "question_count": len(manifest["rows"]), "pair_count": len(records),\n        "test_analysis": "blocked until FROZEN.json and owner HALT 2 approval"})\n    return cfg, index, environment\n\n\ndef evaluate_sealed_once(root, cfg, manifest, index, environment, owner_ok_halt2=False):\n    if owner_ok_halt2 is not True:\n        raise RuntimeError("HALT 2: owner must approve frozen choices before test evaluation")\n    out = Path(root) / cfg.model\n    receipt = out / "test_evaluation.json"\n    if receipt.exists():\n        saved = avh.read_json(receipt)\n        if (saved["metrics_sha256"] != avh.file_hash(out / "metrics.json")\n                or saved["freeze_sha256"] != avh.file_hash(out / "FROZEN.json")):\n            raise RuntimeError("Frozen test result changed")\n        return avh.read_json(out / "metrics.json")\n    if (out / "metrics.json").exists():\n        raise RuntimeError("Unreceipted metrics exist; inspect before evaluating again")\n    result = avh.evaluate_test(root, cfg, manifest, index, environment, "GO")\n    mde_path = out / "calibration_mde.json"\n    if mde_path.exists():\n        mde = avh.read_json(mde_path)\n        result["calibration_mde_planning"] = mde\n        if any(r.get("mde_delta_auroc_approx") is not None and r["mde_delta_auroc_approx"] > .05\n               for r in mde["tasks"].values()):\n            result["limitations"] = [*result.get("limitations", []),\n                "Calibration-based approximate MDE exceeds 0.05 for at least one primary task"]\n        avh.atomic_json(out / "metrics.json", result)\n    immutable_json(receipt, {"status": "evaluated_once", "metrics_sha256": avh.file_hash(out / "metrics.json"),\n                           "freeze_sha256": avh.file_hash(out / "FROZEN.json")})\n    return result\n\n\ndef calibration_mde(calibration):\n    """Planning approximation from honest group-out-of-fold calibration predictions.\n\n    Fixed last layer/C=.1 avoids selecting this planning number on validation.\n    CI width is not a power simulation; the assumed normal approximation and\n    alpha/power are explicit. No validation or test rows are accepted.\n    """\n    import numpy as np\n    from sklearn.model_selection import GroupKFold\n    if any(r["split"] != "calibration" for r in calibration):\n        raise RuntimeError("MDE planning must use calibration only")\n    output = {"kind": "calibration_planning_approximation_not_test_result", "alpha": .05,\n        "assumed_power": .8, "layer": "last", "C": .1, "tasks": {},\n        "method": "(1.96+0.842)*CI_width/(2*1.96), paired group-bootstrap OOF Delta AUROC; approximate, not simulated power"}\n    for task in avh.TASKS[:2]:\n        rows = [r for r in calibration if r["task"] == task]\n        yes = [r for r in rows if r["margin"] > 0]\n        groups = np.array([r["source_group"] for r in rows])\n        false_count = sum(not r["label"] for r in yes)\n        summary = {"n_calibration": len(rows), "n_yes": len(yes), "n_false_yes": false_count,\n                   "power_flag": "underpowered" if false_count < 40 else "at_least_40_false_yes",\n                   "mde_delta_auroc_approx": None}\n        output["tasks"][task] = summary\n        if len(set(groups)) < 3 or len({r["label"] for r in yes}) < 2:\n            summary["status"] = "undefined_missing_groups_or_class"\n            continue\n        predictions = {}\n        failed = False\n        for train_i, hold_i in GroupKFold(n_splits=min(5, len(set(groups)))).split(rows, groups=groups):\n            train = [rows[i] for i in train_i]\n            train_yes = [r for r in train if r["margin"] > 0]\n            hold_yes = [rows[i] for i in hold_i if rows[i]["margin"] > 0]\n            if not hold_yes:\n                continue\n            if len({r["label"] for r in train_yes}) < 2:\n                failed = True\n                break\n            y = np.array([1-r["label"] for r in train_yes])\n            bx = avh.baseline_matrix(train, train_yes, training=True)\n            hx = avh.baseline_matrix(train, hold_yes)\n            transform = avh.Residualizer().fit(train, train[0]["hidden"].shape[0] - 1)\n            base = avh.logistic(bx, y, .1)\n            aug = avh.logistic(np.column_stack([bx, transform.transform(train_yes)]), y, .1)\n            bp = base.predict_proba(hx)[:, 1]\n            ap = aug.predict_proba(np.column_stack([hx, transform.transform(hold_yes)]))[:, 1]\n            predictions.update({r["id"]: (float(a), float(b)) for r, a, b in zip(hold_yes, ap, bp)})\n        if failed or len(predictions) != len(yes):\n            summary["status"] = "undefined_oof_training_class"\n            continue\n        paired = np.array([[1-r["label"], *predictions[r["id"]]] for r in yes])\n        def delta(a):\n            first, second = avh.safe_auc(a[:, 0], a[:, 1]), avh.safe_auc(a[:, 0], a[:, 2])\n            return first-second if first is not None and second is not None else None\n        ci = avh.cluster_ci(paired, [r["source_group"] for r in yes], delta)\n        summary["oof_delta_auroc"] = ci\n        if ci["ci95"] is None:\n            summary["status"] = "undefined_bootstrap_width"\n            continue\n        summary["mde_delta_auroc_approx"] = (1.96+.842) * (ci["ci95"][1]-ci["ci95"][0]) / (2*1.96)\n        summary["status"] = "planning_approximation"\n        summary["sensitivity_limitation_gt_005"] = summary["mde_delta_auroc_approx"] > .05\n    return output\n\n\ndef validation_report(root, cfg, manifest, index, environment):\n    """Descriptive validation numbers after choices have been frozen; no test rows."""\n    import joblib\n    import numpy as np\n    out = Path(root) / cfg.model\n    freeze = avh.read_json(out / "FROZEN.json")\n    if avh.file_hash(out / "analysis_bundle.joblib") != freeze["bundle_sha256"]:\n        raise RuntimeError("Frozen analysis state changed")\n    saved = joblib.load(out / "analysis_bundle.joblib")  # Locally created trusted state only.\n    rows = avh.load_completed(root, cfg, manifest, index, environment, "validation")\n    result = {"split": "validation", "choices_frozen": True, "tasks": {}}\n    for task in avh.TASKS[:2]:\n        selected = [r for r in rows if r["task"] == task]\n        y, g = [r["label"] for r in selected], [r["source_group"] for r in selected]\n        entry = {"full": avh.rates(y, [r["margin"] > 0 for r in selected], g)}\n        bundle = saved["bundles"][task]\n        yes = [r for r in selected if r["margin"] > 0]\n        if bundle["status"] == "fitted" and len({r["label"] for r in yes}) == 2:\n            scores = avh.probe_scores(bundle, saved["calibration"], yes)\n            labels = np.array([1-r["label"] for r in yes])\n            entry["baseline_auc"] = avh.auc_ci(labels, scores["baseline"], [r["source_group"] for r in yes])\n            entry["augmented_auc"] = avh.auc_ci(labels, scores["augmented"], [r["source_group"] for r in yes])\n        else:\n            entry["probe_status"] = "underpowered_missing_class"\n        result["tasks"][task] = entry\n    avh.atomic_json(out / "validation_results.json", result)\n    return result\n\n\ndef launch_account(runtime, job, output, hf_token=None):\n    """Two new Python processes; CUDA is never initialized in the notebook parent."""\n    workers = account_workers(job["account_index"], job["n_workers"])\n    if job.get("mode") not in ("timing", "auto", "run", "addon"):\n        raise ValueError("Unknown worker mode")\n    if job.get("reference") != "fresh_unbatched_gpu":\n        raise RuntimeError("Reference decision pending; no GPU load started")\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    import uuid\n    launch_id = uuid.uuid4().hex\n    gate_path = output / f"gate-{launch_id}.json"\n    worker_outputs = [output.parent / f"out_{job[\'account_index\']}_{worker}" for worker in workers]\n    children, handles = [], []\n    gate_written, calibration_printed = False, False\n    try:\n        for gpu, worker in enumerate(workers):\n            cfg = dict(job, worker_id=worker, local_gpu=gpu, output=str(worker_outputs[gpu]),\n                       launch_id=launch_id, gate_file=str(gate_path))\n            job_path = output / f"worker-{worker:02d}-job.json"\n            avh.atomic_json(job_path, cfg)  # No credentials in this record.\n            log = (output / f"worker-{worker:02d}.log").open("w", encoding="utf-8")\n            handles.append(log)\n            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), TOKENIZERS_PARALLELISM="false")\n            if hf_token:\n                env["HF_TOKEN"] = hf_token\n            children.append(subprocess.Popen([sys.executable, str(Path(runtime) / "avh_kaggle_worker.py"),\n                "--job", str(job_path)], env=env, stdout=log, stderr=subprocess.STDOUT))\n        last = 0\n        while any(p.poll() is None for p in children):\n            if not calibration_printed:\n                receipts = [avh.read_json(p / "calibration_receipt.json") for p in worker_outputs\n                            if (p / "calibration_receipt.json").exists()]\n                receipts = [r for r in receipts if r["launch_id"] == launch_id]\n                if len(receipts) == 2:\n                    print("CALIBRATION_ACCOUNT_JSON=" + json.dumps(receipts), flush=True)\n                    avh.atomic_json(output.parent / "calibration_account.json", receipts)\n                    calibration_printed = True\n            if job["mode"] == "auto" and not gate_written:\n                reports = [avh.read_json(p / "timing.json") for p in worker_outputs if (p / "timing.json").exists()]\n                reports = [r for r in reports if r.get("launch_id") == launch_id]\n                if len(reports) == 2:\n                    summary = timing_summary(reports, avh.read_json(job["manifest"]), job["n_workers"])\n                    startup = max(r["load_seconds"] + r["smoke_seconds"] for r in reports)\n                    elapsed = time.monotonic() - job["session_started"]\n                    # Friends start after receiving this approval. Include their observed startup allowance.\n                    total = summary["forecast"]["ideal_wall_hours_scoring_only"] + (elapsed + startup) / 3600\n                    summary["forecast"]["wall_hours_including_measured_owner_elapsed_and_friend_startup"] = total\n                    approved = total <= 2.5\n                    avh.atomic_json(output.parent / "timing_summary.json", summary)\n                    gate = {"status": "approved" if approved else "halt_forecast",\n                        "bundle_sha256": reports[0]["identity"]["bundle_sha256"],\n                        "code_sha256": reports[0]["identity"]["code_sha256"],\n                        "code_git_hash": reports[0]["identity"]["code_git_hash"],\n                        "code_dirty": reports[0]["identity"]["code_dirty"],\n                        "identity_sha256": avh.digest(reports[0]["identity"]),\n                        "n_workers": job["n_workers"], "n": 2000,\n                        "numerical_smoke_passed": True,\n                        "comparison_kind": reports[0]["identity"].get("comparison_kind", "batch_parity"),\n                        "forecast_wall_hours_planning_only": total,\n                        "timing_summary_sha256": avh.digest(summary)}\n                    avh.atomic_json(gate_path, gate)\n                    avh.atomic_json(output.parent / "RUN_APPROVAL.json", gate)\n                    print("RUN_APPROVAL_JSON=" + json.dumps(gate), flush=True)\n                    print(json.dumps(summary, indent=2), flush=True)\n                    print((Path(job["dataset"]) / "runtime/PREREGISTRATION_KAGGLE.md").read_text(), flush=True)\n                    if "serial_amendment_sha256" in reports[0]["identity"]:\n                        print(Path(__file__).with_name("GEMMA_SERIAL_AMENDMENT.md").read_text(), flush=True)\n                    if not approved:\n                        print("HALT: forecast exceeds 2.5 h. Proposed cut for owner review: test 1200 -> 900; keep cal/val and every pass.", flush=True)\n                    gate_written = True\n            if any(p.poll() not in (None, 0) for p in children):\n                raise RuntimeError("A worker failed; inspect private worker logs")\n            if time.monotonic() - last > 30:\n                print("Workers running:", [w for w, p in zip(workers, children) if p.poll() is None], flush=True)\n                last = time.monotonic()\n            time.sleep(2)\n        if any(p.returncode for p in children):\n            raise RuntimeError("Worker failed; inspect private logs")\n    finally:\n        for p in children:\n            if p.poll() is None:\n                p.terminate()\n                try:\n                    p.wait(timeout=20)\n                except subprocess.TimeoutExpired:\n                    p.kill()\n                    p.wait()\n        for handle in handles:\n            handle.close()\n', 'avh_kaggle_worker.py': '"""Fresh-process GPU worker. CUDA_VISIBLE_DEVICES is set by the notebook parent.\n\nNo account login automation. No test statistics. One model instance on local cuda:0.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport collections\nimport gc\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport time\n\nimport avh_common as avh\nimport avh_kaggle as kg\n\n\ndef guard_empty_sdpa_queries(mask_factory, bool_dtype):\n    """Keep fully masked padding queries finite so they cannot contaminate real tokens."""\n    def guarded(*args, **kwargs):\n        mask = mask_factory(*args, **kwargs)\n        if mask is not None and mask.dtype == bool_dtype:\n            return mask | ~mask.any(dim=-1, keepdim=True)\n        return mask\n    return guarded\n\n\nclass BatchScorer:\n    """Fresh full forwards with native processor padding; never prefix-cache reuse."""\n    def __init__(self, scorer):\n        self.scorer = scorer\n        self.torch = scorer.torch\n        scorer.processor.tokenizer.padding_side = "left"\n\n    def score(self, row, names, index, diagnostic=False):\n        import numpy as np\n        torch, s = self.torch, self.scorer\n        torch.cuda.synchronize()\n        tick = time.monotonic()\n        texts, images, image_groups, audios = [], [], [], []\n        for name in names:\n            ims, audio, _ = s.media(row, name, index)\n            content = ([{"type": "image"} for _ in ims] + [{"type": "audio"}]) if ims else []\n            content += [{"type": "text", "text": "\\nQuestion: " + row["question"] + "\\nAnswer Yes or No only."}]\n            texts.append(s.processor.apply_chat_template([{"role": "user", "content": content}],\n                tokenize=False, add_generation_prompt=True, enable_thinking=False))\n            # Gemma validates one image group per text row, including empty groups\n            # for no-media rows. A flat image list means ONE sample, not B samples.\n            image_groups.append(list(ims) if ims else [])\n            if ims:\n                images.extend(ims)\n                audios.append(audio)\n        args = {"text": texts, "padding": True, "return_tensors": "pt"}\n        if images:\n            args.update(images=image_groups if s.cfg.model == "gemma" else images,\n                        audio=audios, sampling_rate=16000)\n        if s.cfg.model == "qwen":\n            args["use_audio_in_video"] = False\n        encoded = s.processor(**args)\n        inputs = {k: v.to(device="cuda", dtype=torch.float16 if v.is_floating_point() else v.dtype)\n                  for k, v in encoded.items() if isinstance(v, torch.Tensor)}\n        if s.cfg.model == "qwen":\n            lengths = inputs.get("feature_attention_mask")\n            lengths = lengths.sum(-1) if lengths is not None else None\n            positions, _ = s.model.get_rope_index(inputs["input_ids"], inputs.get("image_grid_thw"),\n                None, inputs["attention_mask"], False, lengths, None)\n            inputs["position_ids"] = positions\n            if hasattr(s.model, "rope_deltas"):\n                s.model.rope_deltas = None\n        else:\n            inputs["position_ids"] = (inputs["attention_mask"].long().cumsum(-1) - 1).clamp_min(0)\n        need_hidden = diagnostic or "full" in names\n        with torch.inference_mode():\n            output = s.model(**inputs, use_cache=False, return_dict=True, output_hidden_states=need_hidden)\n        logits = output.logits[:, -1].float()\n        if not torch.isfinite(logits).all():\n            raise FloatingPointError(f"Nonfinite fp16 logits for passes {names}; padding={s.processor.tokenizer.padding_side}")\n        hidden = torch.stack([h[:, -1].float() for h in output.hidden_states], dim=1) if need_hidden else None\n        if hidden is not None and (not torch.isfinite(hidden).all() or not torch.isfinite(hidden.half()).all()):\n            raise FloatingPointError("Nonfinite/overflowed fp16 hidden states")\n        yes = torch.logsumexp(logits[:, s.tokens["yes"]], -1)\n        no = torch.logsumexp(logits[:, s.tokens["no"]], -1)\n        result = []\n        for i, name in enumerate(names):\n            margin = float(yes[i] - no[i])\n            result.append({"pass": name, "margin": margin, "answer": int(margin > 0),\n                "yes_logsumexp": float(yes[i]), "no_logsumexp": float(no[i]),\n                "peak_vram_bytes": torch.cuda.max_memory_allocated(),\n                "hidden": hidden[i].cpu().numpy() if hidden is not None and (name == "full" or diagnostic) else None,\n                "logits": logits[i].cpu().numpy() if diagnostic else None})\n        del output, inputs, encoded, logits, hidden, yes, no\n        torch.cuda.synchronize()\n        elapsed = time.monotonic() - tick\n        return result, elapsed\n\n\ndef check_pair(reference, candidate):\n    import numpy as np\n    ld = float(np.max(np.abs(reference["logits"] - candidate["logits"])))\n    hd = float(np.max(np.abs(reference["hidden"] - candidate["hidden"])))\n    if ld > .02 or hd > .02 or reference["answer"] != candidate["answer"]:\n        raise RuntimeError(f"Batch/fresh parity failed: logits={ld}, hidden={hd}")\n    return {"logits_max_abs_diff": ld, "hidden_max_abs_diff": hd}\n\n\ndef question_batches(batch, row, index, size, names=None, diagnostic=False):\n    names = list(names or kg.passes_for(row))\n    result, timings = [], []\n    for start in range(0, len(names), size):\n        part = names[start:start + size]\n        outputs, seconds = batch.score(row, part, index, diagnostic=diagnostic)\n        result.extend(outputs)\n        timings.append({"passes": part, "seconds": seconds, "batch_size": len(part)})\n    return result, timings\n\n\ndef startup_smoke(batch, calibration, index, worker_id):\n    """Check full passes on ten questions and all required passes on five."""\n    s, torch = batch.scorer, batch.torch\n    serial = s.cfg.model == "gemma"\n    items = sorted(calibration, key=lambda r: (r["task"] == avh.TASKS[2],\n                   avh.digest([avh.SEED, "worker-smoke", worker_id, r["id"]])))[:10]\n    if len(items) != 10:\n        raise RuntimeError("Need 10 calibration items for the worker smoke")\n    repeat, references = [], {}\n    for i, row in enumerate(items):\n        first, _ = batch.score(row, ["full"], index, diagnostic=True)\n        second, _ = batch.score(row, ["full"], index, diagnostic=True)\n        repeat.append(check_pair(first[0], second[0]))\n        if i < 10:\n            generated = s.generation_check(row, index)\n            if generated["generated_answer"] != first[0]["answer"]:\n                raise RuntimeError("Generate/scoring disagreement")\n        if i < 5:\n            refs = {}\n            for name in kg.passes_for(row):\n                refs[name] = batch.score(row, [name], index, diagnostic=True)[0][0]\n            references[row["id"]] = refs\n    candidates, accepted = {}, []\n    for size in ((1,) if serial else range(1, 6)):\n        try:\n            diffs = []\n            for row in items[:5]:\n                outputs, _ = question_batches(batch, row, index, size, diagnostic=True)\n                diffs.extend(check_pair(references[row["id"]][r["pass"]], r) for r in outputs)\n            candidates[str(size)] = {"status": "passed", "comparisons": len(diffs),\n                "max_logits_diff": max(r["logits_max_abs_diff"] for r in diffs),\n                "max_hidden_diff": max(r["hidden_max_abs_diff"] for r in diffs)}\n            accepted.append(size)\n        except torch.cuda.OutOfMemoryError:\n            if serial:\n                raise\n            gc.collect()\n            torch.cuda.empty_cache()\n            candidates[str(size)] = {"status": "out_of_memory"}\n    if not accepted:\n        raise RuntimeError("No fitting parity-verified batch size")\n    return {"passed": True, "n_questions": 10, "generation_items": 10,\n        "reference_items": 5, "reference_kind": "fresh_unbatched_gpu",\n        "prefix_cache_verified": False, "comparison_kind": (\n            "serial_full_forward_repeatability" if serial else "full_forward_repeatability_and_batch_parity"),\n        "batch_size": max(accepted), "validated_batch_sizes": accepted,\n        "repeatability": repeat, "candidates": candidates}\n\n\ndef stable_environment(scorer):\n    """Hash numerical runtime, retain the complete raw package list as separate provenance."""\n    import importlib.metadata\n    names = ("torch", "transformers", "numpy", "Pillow", "av", "soundfile", "librosa", "accelerate")\n    result = {k: scorer.env[k] for k in ("python", "torch", "transformers", "gpu", "cuda", "attention_backend")}\n    result["packages"] = {name: importlib.metadata.version(name) for name in names}\n    result.update(scoring_mode="full_forward", fp16_reduced_precision_reduction=\n                  scorer.torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction)\n    return result\n\n\ndef worker(job):\n    # Import torch only here, after this fresh process has its GPU environment.\n    import torch\n    if os.environ.get("CUDA_VISIBLE_DEVICES") != str(job["local_gpu"]):\n        raise RuntimeError("Worker GPU assignment mismatch")\n    if torch.cuda.device_count() != 1 or "T4" not in torch.cuda.get_device_name(0):\n        raise RuntimeError("Exactly one visible T4 per worker required")\n    if job["reference"] != "fresh_unbatched_gpu":\n        raise RuntimeError("Reference decision pending")\n    dataset, out = Path(job["dataset"]), Path(job["output"])\n    out.mkdir(parents=True, exist_ok=True)\n    bundle = avh.read_json(dataset / "kaggle_bundle.json")\n    verified = avh.read_json(job["cpu_verified"])\n    if verified["status"] != "cpu_verified" or verified["bundle_sha256"] != avh.digest(bundle):\n        raise RuntimeError("Run the CPU bundle verification first")\n    if verified["media_index_sha256"] != avh.digest(avh.read_json(dataset / "kaggle_media_index.json")):\n        raise RuntimeError("Relocatable media index changed after CPU verification")\n    model = job["model"]\n    if model == "qwen" and (not job.get("gemma_complete") or job.get("qwen_attempt") != 1):\n        raise RuntimeError("Qwen runs only once on spare workers after Gemma completes")\n    manifest = avh.read_json(job["manifest"])\n    if manifest["model"] != model or manifest["parent_manifest_sha256"] != bundle["parent_manifest_sha256"]:\n        raise RuntimeError("Wrong sample/study family")\n    if job["mode"] in ("run", "addon"):\n        gate = job.get("approval", {})\n        if (gate.get("status") != "approved" or gate.get("bundle_sha256") != avh.digest(bundle)\n                or gate.get("n_workers") != job["n_workers"]):\n            raise RuntimeError("Owner timing auto-gate is missing or mismatched")\n    work = out / "study_evidence"\n    work.mkdir(exist_ok=True)\n    serial_amendment = Path(__file__).with_name("GEMMA_SERIAL_AMENDMENT.md")\n    if model == "gemma":\n        shutil.copyfile(serial_amendment, work / serial_amendment.name)\n    for name in ("terms_review.json", "release_inventory.json", "gemma_LICENSE.txt", "qwen_LICENSE.txt"):\n        shutil.copyfile(dataset / name, work / name)\n    (work / model).mkdir(exist_ok=True)\n    for name in ("model_metadata.json", "scoring_amendment.json"):\n        shutil.copyfile(dataset / model / name, work / model / name)\n    amendment = avh.read_json(dataset / model / "scoring_amendment.json")\n    shutil.copyfile(dataset / model / amendment["prefix_evidence_file"], work / model / amendment["prefix_evidence_file"])\n    avh.check_terms(work, model)\n    meta = avh.read_json(dataset / model / "model_metadata.json")\n    cfg = avh.Config(str(work), model, meta["revision"], k=2, workers=job["n_workers"],\n        worker_index=job["worker_id"], scoring_mode="full_forward")\n    avh.record_scoring_amendment(cfg)\n    index = kg.resolved_index(dataset)\n    started = time.monotonic()\n    deadline = started + (1800 if model == "qwen" else 10.5 * 3600)\n    calibration = [r for r in manifest["rows"] if r["split"] == "calibration"]\n    with avh.Watchdog(3600, deadline=deadline) as dog:\n        if model == "gemma":\n            # This registry override lives only in the dedicated Gemma worker.\n            from transformers.masking_utils import ALL_MASK_ATTENTION_FUNCTIONS\n            masks = ALL_MASK_ATTENTION_FUNCTIONS\n            masks["sdpa"] = guard_empty_sdpa_queries(masks["sdpa"], torch.bool)\n        scorer = avh.OmniScorer(cfg)\n        load_seconds = time.monotonic() - started\n        avh.atomic_json(out / "raw_environment.json", scorer.env)\n        batch = BatchScorer(scorer)\n        dog.touch()\n        smoke_start = time.monotonic()\n        smoke = startup_smoke(batch, calibration, index, job["worker_id"])\n        smoke_seconds = time.monotonic() - smoke_start\n        if model == "qwen" and time.monotonic() - started > 1800:\n            raise RuntimeError("Qwen one-attempt load/smoke gate exceeded; replication pending")\n        avh.atomic_json(out / "smoke.json", smoke)\n        dog.deadline = started + 10.5 * 3600\n        dog.seconds = 600\n        dog.touch()\n        identity = {"schema": kg.SCHEMA, "bundle_sha256": avh.digest(bundle),\n            "selection_family": {k: manifest[k] for k in ("parent_manifest_sha256", "group_assignment_sha256", "seed", "k")},\n            "model": model, "model_revision": cfg.revision, "n_workers": cfg.workers,\n            "code_git_hash": job.get("code_git_hash", bundle["code_git_hash"]), "code_dirty": job.get("code_dirty", True),\n            "code_sha256": avh.digest({name: avh.file_hash(Path(__file__).parent / name)\n                                       for name in ("avh_common.py", "avh_kaggle.py", "avh_kaggle_worker.py")}),\n            "environment": stable_environment(scorer), "reference": job["reference"],\n            "numerical_limit": .02, "scoring_mode": "full_forward",\n            "sdpa_empty_query_guard": model == "gemma"}\n        if model == "gemma":\n            identity["serial_amendment_sha256"] = avh.file_hash(serial_amendment)\n            identity["comparison_kind"] = smoke["comparison_kind"]\n            identity["batch_size"] = smoke["batch_size"]\n        if job["mode"] in ("timing", "auto"):\n            # Engineering-only: 50 owner-account questions split 25/25 over its two GPUs.\n            items = kg.stratified_take(calibration, 50, avh.SEED)\n            items = items[job["local_gpu"]::2]\n            timings = []\n            for row in items:\n                outputs, parts = question_batches(batch, row, index, smoke["batch_size"])\n                write_start = time.monotonic()\n                # Real fp16 serialization is timed; this diagnostic never enters the research journal.\n                full = next(r for r in outputs if r["pass"] == "full")\n                import numpy as np\n                np.save(out / "timing_hidden.npy", full["hidden"].astype(np.float16))\n                with (out / "timing_pairs.jsonl").open("a") as f:\n                    for r in outputs:\n                        f.write(json.dumps({k: v for k, v in r.items() if k not in ("hidden", "logits")}) + "\\n")\n                    f.flush()\n                    os.fsync(f.fileno())\n                write_seconds = time.monotonic() - write_start\n                timings.append({"question_id": row["id"], "source_group": row["source_group"],\n                    "task": row["task"], "batches": parts, "checkpoint_seconds": write_seconds,\n                    "question_seconds": sum(p["seconds"] for p in parts) + write_seconds})\n                dog.touch()\n            avh.atomic_json(out / "timing.json", {"kind": "engineering_timing_not_research_results",\n                "model": model, "worker_id": job["worker_id"], "identity": identity,\n                "launch_id": job["launch_id"], "load_seconds": load_seconds, "smoke_seconds": smoke_seconds,\n                "batch_size": smoke["batch_size"], "rows": timings})\n            (out / "timing_hidden.npy").unlink(missing_ok=True)\n        if job["mode"] == "auto":\n            wait_start = time.monotonic()\n            while not Path(job["gate_file"]).exists():\n                if time.monotonic() - wait_start > 600:\n                    raise RuntimeError("Timing aggregation did not produce an auto-gate within 10 minutes")\n                time.sleep(1)\n            gate = avh.read_json(job["gate_file"])\n            if gate["status"] != "approved":\n                raise RuntimeError("Timing auto-gate halted; inspect report before any bulk run")\n            dog.touch()\n        if job["mode"] != "timing":\n            gate = avh.read_json(job["gate_file"]) if job["mode"] == "auto" else job["approval"]\n            if gate["code_sha256"] != identity["code_sha256"]:\n                raise RuntimeError("Worker code differs from the owner timing gate")\n            if gate["identity_sha256"] != avh.digest(identity):\n                raise RuntimeError("Worker data/revision/numerical environment differs from the owner timing gate")\n            splits = ("calibration", "validation", "test")\n            items = [r for r in manifest["rows"] if r["split"] in splits\n                     and kg.worker_for(r["id"], cfg.workers) == cfg.worker_index]\n            split_order = {"calibration": 0, "validation": 1, "test": 2}\n            items.sort(key=lambda r: (split_order[r["split"]], r["video_id"], r["id"]))\n            if job["mode"] == "addon":\n                initial_ids = {r["id"] for r in avh.read_json(dataset / "lean_manifest_n2000.json")["rows"]}\n                items = [r for r in items if r["id"] not in initial_ids]\n            status = "complete"\n            with kg.PairJournal(out / "predictions", identity, cfg.worker_index, job["account_index"], cfg.workers, manifest) as journal:\n                receipt_written = False\n                for row in items:\n                    if not receipt_written and manifest["n"] == 2000:\n                        receipt = kg.calibration_receipt(journal, manifest, cfg.worker_index, cfg.workers, job["launch_id"])\n                        if receipt is not None:\n                            avh.atomic_json(out / "calibration_receipt.json", receipt)\n                            receipt_written = True\n                    names = [name for name in kg.passes_for(row) if (row["id"], name) not in journal.done]\n                    if not names:\n                        continue\n                    # Exit normally before Kaggle\'s session cap so output version can finish saving.\n                    if time.monotonic() > started + 10 * 3600:\n                        status = "checkpointed_session_budget"\n                        break\n                    outputs, _ = question_batches(batch, row, index, smoke["batch_size"], names=names)\n                    for r in outputs:\n                        journal.append(row, r["pass"], {k: v for k, v in r.items()\n                            if k not in ("pass", "hidden", "logits")}, hidden=r["hidden"])\n                    journal.question_complete()\n                    if journal.dirty_questions == 0:\n                        print(json.dumps({"worker_id": cfg.worker_index, "completed_pairs": len(journal.done)}), flush=True)\n                    dog.touch()\n                if not receipt_written and manifest["n"] == 2000:\n                    receipt = kg.calibration_receipt(journal, manifest, cfg.worker_index, cfg.workers, job["launch_id"])\n                    if receipt is not None:\n                        avh.atomic_json(out / "calibration_receipt.json", receipt)\n            avh.atomic_json(out / "completion.json", {"status": status, "mode": job["mode"],\n                "manifest_sha256": avh.digest(manifest), "worker_id": cfg.worker_index,\n                "test_analysis": "sealed until CPU FROZEN.json and owner HALT 2 approval"})\n        weights = scorer.model.get_output_embeddings().weight\n        direction = (weights[scorer.tokens["yes"]].float().mean(0) - weights[scorer.tokens["no"]].float().mean(0))\n        import numpy as np\n        np.save(out / "yes_no_unembedding.npy", direction.detach().cpu().numpy())\n    print(json.dumps({"worker_id": job["worker_id"], "mode": job["mode"], "finished": True}), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--job", required=True)\n    args = parser.parse_args()\n    worker(avh.read_json(args.job))\n', 'PREREGISTRATION_KAGGLE.md': '# CounterPrior v2: private Kaggle continuation\n\nWritten 2026-11-05 before any lean-study calibration results are inspected. This\namends execution and scope under the owner\'s Kaggle A-to-Z master prompt. The\noriginal Colab files, 5,293-question manifest and 2,092 media clips remain backups.\nNo research outcome is reported here. Qwen replication is pending and outside the\ninitial Gemma timing window. Historical LLaVA claims remain retracted.\n\n## Fixed sample, evidence and passes\n\nGemma: initial n=2,000, calibration 400 / validation 400 / test 1,200. Seed 20261103.\nAssign connected recovered-source groups (or disclosed released-clip fallback)\nonce to 20/20/60 candidate pools. Sample inside each pool by task x yes/no label,\nproportional largest remainder and seeded question-hash order. Preserve original\nsplits as provenance. New split/sample manifests are separate immutable records.\nNo source group crosses splits. Original-source disjointness is unverified when\nlineage is unrecoverable; swapped released clips may share sources across splits.\n\nUse the existing eight frames and complete 16 kHz mono audio. Do not regenerate,\ntruncate or resample them differently. Native processor padding handles different\naudio lengths and image shapes; swaps are not assumed to have equal token length.\nK=2: full, cf0, cf1, blank, no_media. Matching gets full/no_media only. Donors use\nthe existing seeded clip/task rule, first two of four candidate draws, recomputed\ninside the new group-disjoint pools. Donor real/swap status may be unverified;\nretain the existing explicit limitation. Neither blank nor no_media can be cut.\n\n## Predeclared calibration-only add-on\n\nScore the initial sample in one run, calibration then validation then test within\neach of six fixed hash shards. Test predictions are sealed and never analysed\nbefore FROZEN.json. After complete calibration, estimate each primary task\'s\nnegative-label false-yes rate and its grouping-unit bootstrap CI from calibration\nonly. Multiply its point estimate by the initial test-negative inventory solely\nfor sample planning. If either primary task expects fewer than 40 test false-yes\ncases, allow ONE add-on up to n=3,000 (600/600/1,800). Group assignment, seed,\nstratification, donor rule and existing questions remain unchanged. The add-on\nscores only additional pairs. No validation/test prediction may influence this\ndecision, including a decision to stop after the add-on. Actual cells below 40\nfalse-yes cases remain underpowered, even at n=3,000. Analysis uses the final\npredeclared sample. Record the decision before opening sealed test predictions.\n\n## Numerical execution and timing auto-gate\n\nSix worker processes, two per participant, one complete fp16 model per T4.\nCUDA_VISIBLE_DEVICES maps each process to one physical card; never shard weights\nacross cards. Prefix caching failed parity and remains disabled. Question owner is\nstable_hash(question_id) % 6. Participant indices 0/1/2 own workers 0-1/2-3/4-5.\nEach worker runs one 10-question fresh-forward smoke, 10 generation agreements,\nand batch-versus-fresh-unbatched-GPU parity on five calibration questions. Keep\nmax absolute vocabulary-logit and all-layer last-token hidden difference <=0.02,\nidentical answers and finite fp16 states. A fitting batch that fails parity is a\nfailure; do not relax tolerance. Pick the largest fitting, parity-verified batch\nup to all five passes; serialize only when larger batches genuinely do not fit.\n\nOwner timing: 50 stratified calibration questions across its two T4 processes.\nReport observed p50/p90 batch wall time, allocated per-pass time, checkpoint cost,\nmodel loading separately and grouping-unit bootstrap uncertainty where defined.\nAllocated pass time is not an isolated-pass latency measurement. The explicit\nexception to the ban on extrapolated research results is this private **planning\nforecast**, required by the owner: forecast n=2,000 on six workers, label startup,\nparallelism and failure assumptions, never publish it as an experimental result.\nAuto-proceed only if parity passes and forecast <=2.5 hours. Otherwise halt; for\nan excessive forecast propose one owner-reviewed cut: test 1,200 -> 900 (total\n1,700), preserving calibration, validation, blank and no_media. Do not apply that\ncut automatically. A parity failure needs repair, not a claim that a sample cut\nfixes numerical correctness. The ~2-hour target is not a measured guarantee.\n\n## Statistical choices and CPU-only analysis\n\nRetain the existing diagnosis, probe preprocessing and controls. Primary endpoint:\npaired test Delta AUROC of the probe-augmented model over [answer margin +\ncounterfactual contrast + no-media margin + calibration object yes propensity],\namong Yes answers. Use 2,000 paired source-group bootstrap resamples, fixed seed.\nContrast is full margin minus mean of the two swapped margins. Also report the\nblank-modality AVCD-style baseline. AVCD and AVHBench are prior work, not ours.\nThis is AVCD-style queried-modality contrast, not an implementation claim about\nthe published attentive-mask mechanism.\n\nCalibration fits models; validation selects layer, regularisation and thresholds.\nNo new activation steering method is introduced. Centre features using unlabeled\ncalibration states and residualize against answer margin; retain shuffled-label\nand Yes-minus-No projection controls. Choose policy thresholds on validation.\nShip a demo flag only if the paired false-yes reduction CI excludes zero in the\nbeneficial direction AND the measured recall drop is <=2 percentage points.\nReport every cell\'s false-yes count; fewer than 40 means underpowered. Null wording:\n"No supported improvement was measured"; a CI crossing zero is inconclusive,\nnot proof of equivalence or safety. Missing classes are reported explicitly.\n\nCalibration MDE planning: use source-group out-of-fold predictions at a fixed\nlast layer/C=0.1 (no test or validation choice), paired bootstrap Delta AUROC CI\nwidth, and the explicitly approximate normal formula (1.96+0.842)*SE, with\nSE=CI_width/(2*1.96), two-sided alpha=.05 and assumed power=.80. This is a planning\napproximation, not a simulated power guarantee. If undefined, report undefined;\nif >.05, disclose insufficient sensitivity under limitations. Its numerical value\nis pending calibration; never invent it before the run.\n\nThe complete label audit plus 50 object spot-checks is required before analysis\nfreeze. Audited exclusions apply in analysis, not inference. Write FROZEN.json\nwith model, data, code, audit and analysis-state hashes. HALT 2: owner reviews\nfrozen choices/validation numbers. Evaluate sealed test exactly once afterwards.\nHALT 3: owner reviews test CIs, power flags and ship verdict before site work.\n\n## Storage, privacy and provenance\n\nOne private versioned Kaggle dataset; Drive is retained as backup. No public media,\naudio or frames: public content is aggregates plus our own text only. Each real\nparticipant uses their own accounts and token; credentials never enter files or\nlogs. Save full-only all-layer float16 NPY shards of 250 questions, JSONL per pair,\nflush every 25 completed questions and on normal exit. Abrupt termination can\nrepeat uncommitted work. Resume only from attached saved private output versions;\nskip completed pairs. Stop normally before 11 hours so the commit can save.\nCoverage checks require every final sampled pair exactly once with matching run\nidentity and correct assignment. No partial run becomes metrics.json.\n\nKeep prior-work references from the original reviewed preregistration. Verify\noriginal papers before adding any new novelty claim or public citation. No new\nclaim is made here. This is independent continuation in November 2026, separate\nfrom the November 2025-February 2026 LLaVA internship.\n', 'GEMMA_SERIAL_AMENDMENT.md': '# Gemma serial scoring amendment\n\nThe owner approved serial Gemma scoring after both\nworkers failed batched numerical parity. Worker 00 reported maximum absolute\ndifferences of 0.1796875 for logits and 0.25 for hidden states. Worker 01\nreported 0.1015625 and 0.1875. The preregistered limit is 0.02.\n\nThis amendment supersedes the batching and out-of-memory fallback rules\nfor Gemma. Each pass uses one fresh FP16 forward on a T4 with prefix caching\ndisabled. Each worker checks repeatability on ten full passes and on every\nrequired pass for five calibration questions. The 0.02 limits, finite-value\nchecks, identical answers and ten generation agreements apply to this smoke.\nIts outcome is serial repeatability. The previous batched parity failures\nremain part of the run record.\n\nThe owner times the same 50 stratified calibration questions in serial mode.\nThe owner run can proceed automatically after the serial smoke passes on\nboth workers and the six-worker planning forecast is at most 2.5 hours.\nThe dataset version, pass set and sealed test procedure follow the original\npreregistration. The run identity binds this amendment to the worker code.\n'}
for name, text in payload.items():
    (runtime / name).write_text(text, encoding="utf-8")
sys.path.insert(0, str(runtime))
for name in ("avh_common", "avh_kaggle"):
    sys.modules.pop(name, None)
import avh_common as avh
import avh_kaggle as kg
markers = list(Path("/kaggle/input").rglob("kaggle_bundle.json"))
assert len(markers) == 1, "Attach exactly one private counterprior-v2-data dataset"
DATA = markers[0].parent
bundle = avh.read_json(markers[0])
assert bundle["private_only"] is True
assert (DATA / "runtime/PREREGISTRATION_KAGGLE.md").read_text() == payload["PREREGISTRATION_KAGGLE.md"], "Protocol version mismatch"
print("Private study found:", DATA)


In [ ]:
OWNER_APPROVAL = None
ADDON = False
PROVISIONAL = True
assert N_WORKERS == 6, "Worker count is fixed for this run"
kg.account_workers(ACCOUNT_INDEX, N_WORKERS)
if PROVISIONAL:
    assert ACCOUNT_INDEX in (1, 2), "Provisional notebook is for B and C only"
    print("PROVISIONAL: only Account A's RUN_APPROVAL_JSON authorizes analysis.")
elif OWNER_APPROVAL is None and ACCOUNT_INDEX != 0:
    raise RuntimeError("Wait for the owner's approved notebook link; keep GPU off")
if ADDON and OWNER_APPROVAL is None:
    raise RuntimeError("Add-on needs the owner's original timing approval")
version = avh.read_json(DATA / "gemma/model_metadata.json")["revision"]
saved = DATA / "prior_colab/gemma/pre_smoke.json"
assert saved.is_file(), "Saved Gemma environment missing"
transformers_version = avh.read_json(saved)["environment"]["transformers"]
packages = [f"transformers=={transformers_version}", "accelerate==1.10.1", "numpy==2.2.5",
    "scikit-learn==1.7.1", "Pillow==11.3.0", "av==16.0.1", "soundfile==0.13.1",
    "librosa==0.11.0", "joblib==1.5.2"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)
subprocess.run([sys.executable, "-c", "from transformers import Gemma4ForConditionalGeneration; print('Gemma class verified')"], check=True)
# Parent never imports torch or initializes CUDA. Verify both devices externally.
devices = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).splitlines()
assert len(devices) == 2 and all("T4" in d for d in devices), "Select GPU T4 x2"
for name, expected in bundle["files"].items():
    if not name.startswith("media/"):
        assert avh.file_hash(DATA / name) == expected, f"Metadata changed: {name}"
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("HF_TOKEN")
job = {"n_workers": N_WORKERS, "account_index": ACCOUNT_INDEX, "model": "gemma",
    "mode": "addon" if ADDON else "auto" if OWNER_APPROVAL is None else "run",
    "dataset": str(DATA), "manifest": str(DATA / ("lean_manifest_n3000.json" if ADDON else "lean_manifest_n2000.json")),
    "cpu_verified": str(DATA / "cpu_verified.json"), "reference": "fresh_unbatched_gpu",
    "sample_policy": "calibration_addon", "approval": OWNER_APPROVAL,
    "session_started": SESSION_STARTED,
    "code_git_hash": OWNER_APPROVAL["code_git_hash"] if OWNER_APPROVAL else 'cc0c1a45ff0da68949fdc73c466b989e5822f5e9',
    "code_dirty": OWNER_APPROVAL["code_dirty"] if OWNER_APPROVAL else True}
destination = Path(f"/kaggle/working/account_{ACCOUNT_INDEX}")
# Attach your own previous private notebook output to resume. Never steal another worker's checkpoint.
resume_dirs = [p.parent for p in Path("/kaggle/input").rglob("identity.json")
               if (p.parent / "predictions.jsonl").exists()]
for worker in kg.account_workers(ACCOUNT_INDEX, N_WORKERS):
    matches = []
    for path in resume_dirs:
        first = next(iter((path / "predictions.jsonl").read_text().splitlines()), None)
        if first and json.loads(first)["worker_id"] == worker:
            matches.append(path)
    selected = kg.select_output_versions(matches)
    if len(selected) > 1:
        raise RuntimeError("Attach the latest cumulative checkpoint for this worker only")
    target = Path(f"/kaggle/working/out_{ACCOUNT_INDEX}_{worker}/predictions")
    if selected and not target.exists():
        kg.restore_checkpoint(selected[0], target, worker, avh.digest(bundle))
try:
    kg.launch_account(runtime, job, destination, hf_token=token)
    status = {"status": "finished", "account_index": ACCOUNT_INDEX, "provisional": PROVISIONAL}
except Exception as exc:
    # Finish normally so Kaggle can commit partial checkpoints and diagnostics.
    status = {"status": "blocked_checkpoint_preserved", "account_index": ACCOUNT_INDEX,
              "error": str(exc), "action": "Send private logs to owner; no blind rerun"}
finally:
    del token
avh.atomic_json(Path("/kaggle/working/account_status.json"), status)
print(json.dumps(status, indent=2))
